In [1]:
import CAST
import scanpy as sc
import os
import numpy as np
import warnings
import dgl
import torch
import CAST
import os
import numpy as np
import anndata as ad
import scanpy as sc
import warnings
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from sklearn.linear_model import LinearRegression
warnings.filterwarnings("ignore")

找对应点时在保证距离最近的同时先保证处于同一cluster

In [60]:
aligned_coords_int = pd.read_csv("/p2/zulab/jtian/data/SA/05_CAST/output_allCoordsClustersMerge/aligned_coords_int.csv")

In [61]:
aligned_coords_int

,0-15vs0_x,0-15vs0_y,15-15vs0_x,15-15vs0_y,15-30vs15_x,15-30vs15_y,30-30vs15_x,30-30vs15_y,30-45vs30_x,30-45vs30_y,45-45vs30_x,45-45vs30_y,45-60vs45_x,45-60vs45_y,60-60vs45_x,60-60vs45_y
0,49939.0,5370.0,50494.0,-228.0,43359.0,9530.0,43875.0,9015.0,29839.0,9230.0,29672.0,9620.0,15799.0,10650.0,15297.0,10644.0
1,49959.0,5370.0,50474.0,-228.0,43379.0,9530.0,43895.0,9014.0,29859.0,9230.0,29692.0,9620.0,15819.0,10650.0,15317.0,10644.0
2,49979.0,5370.0,50454.0,-229.0,43399.0,9530.0,43916.0,9012.0,29879.0,9230.0,29712.0,9620.0,15839.0,10650.0,15337.0,10645.0
3,49999.0,5370.0,50434.0,-229.0,43419.0,9530.0,43936.0,9010.0,29899.0,9230.0,29732.0,9620.0,15859.0,10650.0,15357.0,10645.0
4,50019.0,5370.0,50415.0,-230.0,43439.0,9530.0,43956.0,9009.0,29919.0,9230.0,29752.0,9620.0,15879.0,10650.0,15377.0,10645.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53404,NaN,NaN,49903.0,5220.0,43819.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53405,NaN,NaN,49883.0,5220.0,43839.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53406,NaN,NaN,49863.0,5219.0,43859.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53407,NaN,NaN,49843.0,5219.0,43879.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [62]:
adata = ad.read_h5ad("/p2/zulab/jtian/data/SA/05_CAST/output_cast9/demo1_cast9.h5ad") 

In [63]:
mask = adata.obs['sample']=='60'

In [64]:
c = adata.obs.loc[mask,['x','y']].copy()

In [65]:
c

,x,y
SpotIndex,,
196698,1899,10530
196699,1919,10530
196700,1939,10530
196701,1959,10530
196702,1979,10530
...,...,...
247357,3119,4970
247358,3139,4970
247359,3159,4970


In [66]:
coords = pd.concat([aligned_coords_int,c.reset_index(drop=True)],axis=1)

In [67]:
coords

,0-15vs0_x,0-15vs0_y,15-15vs0_x,15-15vs0_y,15-30vs15_x,15-30vs15_y,30-30vs15_x,30-30vs15_y,30-45vs30_x,30-45vs30_y,45-45vs30_x,45-45vs30_y,45-60vs45_x,45-60vs45_y,60-60vs45_x,60-60vs45_y,x,y
0,49939.0,5370.0,50494.0,-228.0,43359.0,9530.0,43875.0,9015.0,29839.0,9230.0,29672.0,9620.0,15799.0,10650.0,15297.0,10644.0,1899.0,10530.0
1,49959.0,5370.0,50474.0,-228.0,43379.0,9530.0,43895.0,9014.0,29859.0,9230.0,29692.0,9620.0,15819.0,10650.0,15317.0,10644.0,1919.0,10530.0
2,49979.0,5370.0,50454.0,-229.0,43399.0,9530.0,43916.0,9012.0,29879.0,9230.0,29712.0,9620.0,15839.0,10650.0,15337.0,10645.0,1939.0,10530.0
3,49999.0,5370.0,50434.0,-229.0,43419.0,9530.0,43936.0,9010.0,29899.0,9230.0,29732.0,9620.0,15859.0,10650.0,15357.0,10645.0,1959.0,10530.0
4,50019.0,5370.0,50415.0,-230.0,43439.0,9530.0,43956.0,9009.0,29919.0,9230.0,29752.0,9620.0,15879.0,10650.0,15377.0,10645.0,1979.0,10530.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53404,NaN,NaN,49903.0,5220.0,43819.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53405,NaN,NaN,49883.0,5220.0,43839.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53406,NaN,NaN,49863.0,5219.0,43859.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53407,NaN,NaN,49843.0,5219.0,43879.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [68]:
coords.columns = [
    '0-original-x','0-original-y',
    '15-changed-x','15-changed-y',
    '15-original-x','15-original-y',
    '30-changed-x','30-changed-y',
    '30-original-x','30-original-y',
    '45-changed-x','45-changed-y',
    '45-original-x','45-original-y',
    '60-changed-x','60-changed-y',
    '60-original-x','60-original-y'
]

In [69]:
coords.to_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/unchainedIntCoordsMerge.csv",index=False)

In [70]:
cell_label_dict = torch.load("/p1/data/jtian/SA/05_CAST/output_castProj1/cell_label_dict_k20.pt", map_location="cpu")

In [71]:
cell_label_dict

{'0': array([ 5,  5,  5, ...,  0,  0, 19], dtype=int32),
 '15': array([ 5,  5,  5, ..., 19, 19,  5], dtype=int32),
 '30': array([5, 5, 5, ..., 5, 5, 5], dtype=int32),
 '45': array([5, 5, 5, ..., 5, 5, 5], dtype=int32),
 '60': array([5, 5, 5, ..., 5, 5, 5], dtype=int32)}

In [72]:
coords

,0-original-x,0-original-y,15-changed-x,15-changed-y,15-original-x,15-original-y,30-changed-x,30-changed-y,30-original-x,30-original-y,45-changed-x,45-changed-y,45-original-x,45-original-y,60-changed-x,60-changed-y,60-original-x,60-original-y
0,49939.0,5370.0,50494.0,-228.0,43359.0,9530.0,43875.0,9015.0,29839.0,9230.0,29672.0,9620.0,15799.0,10650.0,15297.0,10644.0,1899.0,10530.0
1,49959.0,5370.0,50474.0,-228.0,43379.0,9530.0,43895.0,9014.0,29859.0,9230.0,29692.0,9620.0,15819.0,10650.0,15317.0,10644.0,1919.0,10530.0
2,49979.0,5370.0,50454.0,-229.0,43399.0,9530.0,43916.0,9012.0,29879.0,9230.0,29712.0,9620.0,15839.0,10650.0,15337.0,10645.0,1939.0,10530.0
3,49999.0,5370.0,50434.0,-229.0,43419.0,9530.0,43936.0,9010.0,29899.0,9230.0,29732.0,9620.0,15859.0,10650.0,15357.0,10645.0,1959.0,10530.0
4,50019.0,5370.0,50415.0,-230.0,43439.0,9530.0,43956.0,9009.0,29919.0,9230.0,29752.0,9620.0,15879.0,10650.0,15377.0,10645.0,1979.0,10530.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53404,NaN,NaN,49903.0,5220.0,43819.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53405,NaN,NaN,49883.0,5220.0,43839.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53406,NaN,NaN,49863.0,5219.0,43859.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53407,NaN,NaN,49843.0,5219.0,43879.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [73]:
for key, labels in cell_label_dict.items():

    labels_full = np.full(len(coords), np.nan)
    labels_full[:len(labels)] = labels

    col_name = f"{key}-original-x"
    idx = coords.columns.get_loc(col_name)

    coords.insert(idx, f"{key}-label", labels_full)

In [74]:
coords

,0-label,0-original-x,0-original-y,15-changed-x,15-changed-y,15-label,15-original-x,15-original-y,30-changed-x,30-changed-y,...,45-changed-x,45-changed-y,45-label,45-original-x,45-original-y,60-changed-x,60-changed-y,60-label,60-original-x,60-original-y
0,5.0,49939.0,5370.0,50494.0,-228.0,5.0,43359.0,9530.0,43875.0,9015.0,...,29672.0,9620.0,5.0,15799.0,10650.0,15297.0,10644.0,5.0,1899.0,10530.0
1,5.0,49959.0,5370.0,50474.0,-228.0,5.0,43379.0,9530.0,43895.0,9014.0,...,29692.0,9620.0,5.0,15819.0,10650.0,15317.0,10644.0,5.0,1919.0,10530.0
2,5.0,49979.0,5370.0,50454.0,-229.0,5.0,43399.0,9530.0,43916.0,9012.0,...,29712.0,9620.0,5.0,15839.0,10650.0,15337.0,10645.0,5.0,1939.0,10530.0
3,5.0,49999.0,5370.0,50434.0,-229.0,5.0,43419.0,9530.0,43936.0,9010.0,...,29732.0,9620.0,5.0,15859.0,10650.0,15357.0,10645.0,5.0,1959.0,10530.0
4,5.0,50019.0,5370.0,50415.0,-230.0,5.0,43439.0,9530.0,43956.0,9009.0,...,29752.0,9620.0,5.0,15879.0,10650.0,15377.0,10645.0,5.0,1979.0,10530.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53404,NaN,NaN,NaN,49903.0,5220.0,19.0,43819.0,3550.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53405,NaN,NaN,NaN,49883.0,5220.0,19.0,43839.0,3550.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53406,NaN,NaN,NaN,49863.0,5219.0,19.0,43859.0,3550.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53407,NaN,NaN,NaN,49843.0,5219.0,19.0,43879.0,3550.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [75]:
coords.to_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/unchainedIntCoordsMergeLabel.csv",index=False)

In [16]:
coords = pd.read_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/unchainedIntCoordsMergeLabel.csv")

In [76]:
coords.columns

Index(['0-label', '0-original-x', '0-original-y', '15-changed-x',
       '15-changed-y', '15-label', '15-original-x', '15-original-y',
       '30-changed-x', '30-changed-y', '30-label', '30-original-x',
       '30-original-y', '45-changed-x', '45-changed-y', '45-label',
       '45-original-x', '45-original-y', '60-changed-x', '60-changed-y',
       '60-label', '60-original-x', '60-original-y'],
      dtype='object')

In [77]:
coords

,0-label,0-original-x,0-original-y,15-changed-x,15-changed-y,15-label,15-original-x,15-original-y,30-changed-x,30-changed-y,...,45-changed-x,45-changed-y,45-label,45-original-x,45-original-y,60-changed-x,60-changed-y,60-label,60-original-x,60-original-y
0,5.0,49939.0,5370.0,50494.0,-228.0,5.0,43359.0,9530.0,43875.0,9015.0,...,29672.0,9620.0,5.0,15799.0,10650.0,15297.0,10644.0,5.0,1899.0,10530.0
1,5.0,49959.0,5370.0,50474.0,-228.0,5.0,43379.0,9530.0,43895.0,9014.0,...,29692.0,9620.0,5.0,15819.0,10650.0,15317.0,10644.0,5.0,1919.0,10530.0
2,5.0,49979.0,5370.0,50454.0,-229.0,5.0,43399.0,9530.0,43916.0,9012.0,...,29712.0,9620.0,5.0,15839.0,10650.0,15337.0,10645.0,5.0,1939.0,10530.0
3,5.0,49999.0,5370.0,50434.0,-229.0,5.0,43419.0,9530.0,43936.0,9010.0,...,29732.0,9620.0,5.0,15859.0,10650.0,15357.0,10645.0,5.0,1959.0,10530.0
4,5.0,50019.0,5370.0,50415.0,-230.0,5.0,43439.0,9530.0,43956.0,9009.0,...,29752.0,9620.0,5.0,15879.0,10650.0,15377.0,10645.0,5.0,1979.0,10530.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53404,NaN,NaN,NaN,49903.0,5220.0,19.0,43819.0,3550.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53405,NaN,NaN,NaN,49883.0,5220.0,19.0,43839.0,3550.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53406,NaN,NaN,NaN,49863.0,5219.0,19.0,43859.0,3550.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53407,NaN,NaN,NaN,49843.0,5219.0,19.0,43879.0,3550.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [78]:
coords0  = coords.iloc[:, 0:3].copy()
coords15 = coords.iloc[:, 3:8].copy()
coords30 = coords.iloc[:, 8:13].copy()
coords45 = coords.iloc[:, 13:18].copy()
coords60 = coords.iloc[:, 18:23].copy()
coords0  = coords0.sort_values(by='0-label').reset_index(drop=False)
coords15 = coords15.sort_values(by='15-label').reset_index(drop=False)
coords30 = coords30.sort_values(by='30-label').reset_index(drop=False)
coords45 = coords45.sort_values(by='45-label').reset_index(drop=False)
coords60 = coords60.sort_values(by='60-label').reset_index(drop=False)

In [79]:
coords0

,index,0-label,0-original-x,0-original-y
0,42344,0.0,50479.0,830.0
1,35452,0.0,48279.0,1530.0
2,10984,0.0,52079.0,3850.0
3,10985,0.0,52099.0,3850.0
4,10986,0.0,52119.0,3850.0
...,...,...,...,...
53404,53404,NaN,NaN,NaN
53405,53405,NaN,NaN,NaN
53406,53406,NaN,NaN,NaN
53407,53407,NaN,NaN,NaN


In [80]:
coords0  = coords0.rename(columns={'index': '0-index'})
coords15 = coords15.rename(columns={'index': '15-index'})
coords30 = coords30.rename(columns={'index': '30-index'})
coords45 = coords45.rename(columns={'index': '45-index'})
coords60 = coords60.rename(columns={'index': '60-index'})

In [81]:
samples = [0,15,30,45,60]
coords_dict = {
    0: coords0,
    15: coords15,
    30: coords30,
    45: coords45,
    60: coords60
}


for i in samples:
    df = coords_dict[i]
    print(df[f'{i}-label'].unique())



[ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17.
 18. 19. nan]
[ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17.
 18. 19.]
[ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17.
 18. 19. nan]
[ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17.
 18. 19. nan]
[ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17.
 18. 19. nan]


In [82]:
label_dfs = {}

for lab in range(20):   # 0–19
    lab = float(lab)    # 转成浮点数
    
    pieces = []
    
    for s, df in coords_dict.items():
        label_col = f'{s}-label'
        
        part = df[df[label_col] == lab].reset_index(drop=True)
        pieces.append(part)
    
    label_dfs[lab] = pd.concat(pieces, axis=1)  # 自动补NaN

In [83]:
label_dfs[0]

,0-index,0-label,0-original-x,0-original-y,15-index,15-changed-x,15-changed-y,15-label,15-original-x,15-original-y,...,45-changed-y,45-label,45-original-x,45-original-y,60-index,60-changed-x,60-changed-y,60-label,60-original-x,60-original-y
0,42344.0,0.0,50479.0,830.0,14016,51855.0,1469.0,0.0,41939.0,7710.0,...,6149.0,0.0,13939.0,7030.0,26705.0,17657.0,7767.0,0.0,4239.0,7670.0
1,35452.0,0.0,48279.0,1530.0,4581,49744.0,484.0,0.0,44099.0,8730.0,...,6071.0,0.0,14419.0,6950.0,40231.0,17525.0,6492.0,0.0,4099.0,6410.0
2,10984.0,0.0,52079.0,3850.0,4580,49764.0,485.0,0.0,44079.0,8730.0,...,6071.0,0.0,14439.0,6950.0,40232.0,17545.0,6493.0,0.0,4119.0,6410.0
3,10985.0,0.0,52099.0,3850.0,21859,51819.0,2126.0,0.0,41959.0,6990.0,...,6071.0,0.0,14459.0,6950.0,40233.0,17565.0,6493.0,0.0,4139.0,6410.0
4,10986.0,0.0,52119.0,3850.0,9249,50700.0,1001.0,0.0,43119.0,8190.0,...,6071.0,0.0,14479.0,6950.0,28990.0,13735.0,7521.0,0.0,319.0,7450.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6621,NaN,NaN,NaN,NaN,48315,49033.0,4413.0,0.0,44719.0,4410.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6622,NaN,NaN,NaN,NaN,2318,50798.0,236.0,0.0,43039.0,9030.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6623,NaN,NaN,NaN,NaN,48312,49092.0,4414.0,0.0,44659.0,4410.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6624,NaN,NaN,NaN,NaN,1823,50484.0,155.0,0.0,43359.0,9110.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [84]:
with open('/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/label_dfs.pkl', 'wb') as f:
    pickle.dump(label_dfs, f)

In [72]:
with open('/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/label_dfs.pkl', 'rb') as f:
    label_dfs = pickle.load(f)

In [85]:
for key, df in label_dfs.items():
    label_dfs[key] = df.loc[:, ~df.columns.str.contains('label')]

In [86]:
label_dfs[0]

,0-index,0-original-x,0-original-y,15-index,15-changed-x,15-changed-y,15-original-x,15-original-y,30-index,30-changed-x,...,45-index,45-changed-x,45-changed-y,45-original-x,45-original-y,60-index,60-changed-x,60-changed-y,60-original-x,60-original-y
0,42344.0,50479.0,830.0,14016,51855.0,1469.0,41939.0,7710.0,12354.0,41848.0,...,34514.0,27814.0,6149.0,13939.0,7030.0,26705.0,17657.0,7767.0,4239.0,7670.0
1,35452.0,48279.0,1530.0,4581,49744.0,484.0,44099.0,8730.0,36268.0,45130.0,...,35388.0,28291.0,6071.0,14419.0,6950.0,40231.0,17525.0,6492.0,4099.0,6410.0
2,10984.0,52079.0,3850.0,4580,49764.0,485.0,44079.0,8730.0,36269.0,45150.0,...,35389.0,28311.0,6071.0,14439.0,6950.0,40232.0,17545.0,6493.0,4119.0,6410.0
3,10985.0,52099.0,3850.0,21859,51819.0,2126.0,41959.0,6990.0,4592.0,42102.0,...,35390.0,28331.0,6071.0,14459.0,6950.0,40233.0,17565.0,6493.0,4139.0,6410.0
4,10986.0,52119.0,3850.0,9249,50700.0,1001.0,43119.0,8190.0,4591.0,42081.0,...,35391.0,28350.0,6071.0,14479.0,6950.0,28990.0,13735.0,7521.0,319.0,7450.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6621,NaN,NaN,NaN,48315,49033.0,4413.0,44719.0,4410.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6622,NaN,NaN,NaN,2318,50798.0,236.0,43039.0,9030.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6623,NaN,NaN,NaN,48312,49092.0,4414.0,44659.0,4410.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6624,NaN,NaN,NaN,1823,50484.0,155.0,43359.0,9110.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [87]:
label_dfs[0].columns

Index(['0-index', '0-original-x', '0-original-y', '15-index', '15-changed-x',
       '15-changed-y', '15-original-x', '15-original-y', '30-index',
       '30-changed-x', '30-changed-y', '30-original-x', '30-original-y',
       '45-index', '45-changed-x', '45-changed-y', '45-original-x',
       '45-original-y', '60-index', '60-changed-x', '60-changed-y',
       '60-original-x', '60-original-y'],
      dtype='object')

In [89]:
coords=pd.read_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/unchainedIntCoordsMerge.csv")

In [90]:
coords

,0-original-x,0-original-y,15-changed-x,15-changed-y,15-original-x,15-original-y,30-changed-x,30-changed-y,30-original-x,30-original-y,45-changed-x,45-changed-y,45-original-x,45-original-y,60-changed-x,60-changed-y,60-original-x,60-original-y
0,49939.0,5370.0,50494.0,-228.0,43359.0,9530.0,43875.0,9015.0,29839.0,9230.0,29672.0,9620.0,15799.0,10650.0,15297.0,10644.0,1899.0,10530.0
1,49959.0,5370.0,50474.0,-228.0,43379.0,9530.0,43895.0,9014.0,29859.0,9230.0,29692.0,9620.0,15819.0,10650.0,15317.0,10644.0,1919.0,10530.0
2,49979.0,5370.0,50454.0,-229.0,43399.0,9530.0,43916.0,9012.0,29879.0,9230.0,29712.0,9620.0,15839.0,10650.0,15337.0,10645.0,1939.0,10530.0
3,49999.0,5370.0,50434.0,-229.0,43419.0,9530.0,43936.0,9010.0,29899.0,9230.0,29732.0,9620.0,15859.0,10650.0,15357.0,10645.0,1959.0,10530.0
4,50019.0,5370.0,50415.0,-230.0,43439.0,9530.0,43956.0,9009.0,29919.0,9230.0,29752.0,9620.0,15879.0,10650.0,15377.0,10645.0,1979.0,10530.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53404,NaN,NaN,49903.0,5220.0,43819.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53405,NaN,NaN,49883.0,5220.0,43839.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53406,NaN,NaN,49863.0,5219.0,43859.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53407,NaN,NaN,49843.0,5219.0,43879.0,3550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [91]:
def chain_match_one_df_hungarian(df: pd.DataFrame, samples=(0, 15, 30, 45, 60)) -> pd.DataFrame:
    """
    对单个 df 做链式匹配：
    起点 = sample 0 的 original 坐标有效的所有行
    然后依次在 15 / 30 / 45 / 60 的 changed 坐标中，
    用匈牙利算法做“当前所有活跃链”的全局一对一最优匹配。

    返回：
        out_df：与原 df 列名一致的新 DataFrame
                每一行代表一条从 sample 0 出发的链
    """

    df = df.copy()

    def c(s, name):
        return f"{s}-{name}"

    # ---------- 1. 找到 sample 0 的起点 ----------
    o0x, o0y = c(0, "original-x"), c(0, "original-y")
    i0col = c(0, "index")

    start_mask = df[o0x].notna() & df[o0y].notna()
    start_indices = df.index[start_mask].to_numpy()

    if len(start_indices) == 0:
        return pd.DataFrame(columns=df.columns)

    # 输出表：先全部填 NA，后面逐步写入
    out_df = pd.DataFrame(pd.NA, index=start_indices, columns=df.columns)

    # 先把 sample 0 的信息写进去
    if i0col in df.columns:
        out_df.loc[start_indices, i0col] = df.loc[start_indices, i0col].to_numpy()
    out_df.loc[start_indices, o0x] = df.loc[start_indices, o0x].to_numpy()
    out_df.loc[start_indices, o0y] = df.loc[start_indices, o0y].to_numpy()

    # 当前“还在继续往后匹配”的链
    # active_out_idx: out_df 中哪些行目前还是活跃的
    # q: 这些活跃链当前要去匹配的查询点坐标
    active_out_idx = start_indices.copy()
    q = df.loc[start_indices, [o0x, o0y]].to_numpy(dtype=float, copy=True)

    # ---------- 2. 依次做 15 / 30 / 45 / 60 ----------
    for s in samples[1:]:
        if len(active_out_idx) == 0:
            break

        idx_col = c(s, "index")
        cx, cy = c(s, "changed-x"), c(s, "changed-y")
        ox, oy = c(s, "original-x"), c(s, "original-y")

        # 该样本没有 changed 坐标列，就断掉
        if (cx not in df.columns) or (cy not in df.columns):
            break

        # 候选池：该样本中所有 changed 坐标有效的行
        cand_mask = df[cx].notna() & df[cy].notna()
        cand_rows = df.index[cand_mask].to_numpy()

        if len(cand_rows) == 0:
            break

        cand_xy = df.loc[cand_rows, [cx, cy]].to_numpy(dtype=float, copy=True)

        # ---------- 3. 代价矩阵 ----------
        # cost[i, j] = 活跃链 i 的当前查询点 q[i] 到 候选点 j 的 changed 坐标 的平方欧氏距离
        # 用平方距离即可，不影响最优匹配结果，还省一次开方
        cost = cdist(q, cand_xy, metric="sqeuclidean")

        # ---------- 4. 匈牙利算法：求当前这一步的全局最优一对一匹配 ----------
        # row_ind 对应 q 的行号
        # col_ind 对应 cand_xy 的列号
        row_ind, col_ind = linear_sum_assignment(cost)

        # 被成功匹配到的 out_df 行
        matched_out_idx = active_out_idx[row_ind]

        # 被选中的 df 候选行
        matched_df_rows = cand_rows[col_ind]

        # ---------- 5. 把匹配结果写回输出表 ----------
        if idx_col in df.columns:
            out_df.loc[matched_out_idx, idx_col] = df.loc[matched_df_rows, idx_col].to_numpy()
        if cx in df.columns:
            out_df.loc[matched_out_idx, cx] = df.loc[matched_df_rows, cx].to_numpy()
        if cy in df.columns:
            out_df.loc[matched_out_idx, cy] = df.loc[matched_df_rows, cy].to_numpy()
        if ox in df.columns:
            out_df.loc[matched_out_idx, ox] = df.loc[matched_df_rows, ox].to_numpy()
        if oy in df.columns:
            out_df.loc[matched_out_idx, oy] = df.loc[matched_df_rows, oy].to_numpy()

        # ---------- 6. 为下一步准备新的查询点 ----------
        # 下一步查询点用这一步匹配到的 original 坐标
        # 只有 original-x / original-y 都不缺失的，才能继续往后链接
        next_valid_mask = (
            df.loc[matched_df_rows, [ox, oy]].notna().all(axis=1).to_numpy()
            if (ox in df.columns and oy in df.columns)
            else np.zeros(len(matched_df_rows), dtype=bool)
        )

        active_out_idx = matched_out_idx[next_valid_mask]
        next_df_rows = matched_df_rows[next_valid_mask]

        if len(next_df_rows) == 0:
            break

        q = df.loc[next_df_rows, [ox, oy]].to_numpy(dtype=float, copy=True)

    # 如果你想把输出索引改成 0,1,2,... 可以取消下一行注释
    # out_df = out_df.reset_index(drop=True)

    return out_df

In [92]:
matched_label_dfs = {
    label: chain_match_one_df_hungarian(df)
    for label, df in label_dfs.items()
}

In [93]:
path = "/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/matched_label_dfs_all.pkl"

with open(path, "wb") as f:
    pickle.dump(matched_label_dfs, f)

In [94]:
matched_label_dfs[0].to_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/matched_label_dfs_all_0.csv",index=False)

In [95]:
for key, df in matched_label_dfs.items():
    matched_label_dfs[key] = df.loc[:, df.columns.str.contains('original')]

In [96]:
matched_label_dfs[0]

,0-original-x,0-original-y,15-original-x,15-original-y,30-original-x,30-original-y,45-original-x,45-original-y,60-original-x,60-original-y
0,50479.0,830.0,43299.0,8430.0,29419.0,8610.0,15459.0,9650.0,2019.0,9510.0
1,48279.0,1530.0,45499.0,7470.0,31639.0,8050.0,17739.0,8970.0,4299.0,8890.0
2,52079.0,3850.0,41719.0,5110.0,28079.0,5450.0,14299.0,6330.0,599.0,6570.0
3,52099.0,3850.0,41699.0,5110.0,28059.0,5450.0,14279.0,6350.0,599.0,6590.0
4,52119.0,3850.0,41679.0,5090.0,28039.0,5430.0,14219.0,6270.0,559.0,6550.0
...,...,...,...,...,...,...,...,...,...,...
5963,52019.0,2810.0,41759.0,6230.0,28039.0,6370.0,14119.0,7330.0,779.0,7530.0
5964,52059.0,2810.0,41719.0,6230.0,27979.0,6530.0,14119.0,7470.0,699.0,7630.0
5965,48899.0,1430.0,45079.0,7630.0,31019.0,8050.0,17039.0,9050.0,3519.0,9130.0
5966,52039.0,2810.0,41739.0,6250.0,27999.0,6550.0,14139.0,7470.0,719.0,7610.0


In [97]:
path = "/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/matched_label_dfs.pkl"

with open(path, "wb") as f:
    pickle.dump(matched_label_dfs, f)

In [40]:
path = "/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/matched_label_dfs.pkl"

with open(path, "rb") as f:
    matched_label_dfs = pickle.load(f)

In [98]:
matched_label_dfs[0]

,0-original-x,0-original-y,15-original-x,15-original-y,30-original-x,30-original-y,45-original-x,45-original-y,60-original-x,60-original-y
0,50479.0,830.0,43299.0,8430.0,29419.0,8610.0,15459.0,9650.0,2019.0,9510.0
1,48279.0,1530.0,45499.0,7470.0,31639.0,8050.0,17739.0,8970.0,4299.0,8890.0
2,52079.0,3850.0,41719.0,5110.0,28079.0,5450.0,14299.0,6330.0,599.0,6570.0
3,52099.0,3850.0,41699.0,5110.0,28059.0,5450.0,14279.0,6350.0,599.0,6590.0
4,52119.0,3850.0,41679.0,5090.0,28039.0,5430.0,14219.0,6270.0,559.0,6550.0
...,...,...,...,...,...,...,...,...,...,...
5963,52019.0,2810.0,41759.0,6230.0,28039.0,6370.0,14119.0,7330.0,779.0,7530.0
5964,52059.0,2810.0,41719.0,6230.0,27979.0,6530.0,14119.0,7470.0,699.0,7630.0
5965,48899.0,1430.0,45079.0,7630.0,31019.0,8050.0,17039.0,9050.0,3519.0,9130.0
5966,52039.0,2810.0,41739.0,6250.0,27999.0,6550.0,14139.0,7470.0,719.0,7610.0


In [99]:
matched_label_dfs[0].columns

Index(['0-original-x', '0-original-y', '15-original-x', '15-original-y',
       '30-original-x', '30-original-y', '45-original-x', '45-original-y',
       '60-original-x', '60-original-y'],
      dtype='object')

In [100]:
dfs = []

for label, df in matched_label_dfs.items():
    df_copy = df.copy()
    df_copy.insert(0, 'cell-label', label) 
    dfs.append(df_copy)
chainedOCoordsLabel = pd.concat(dfs, ignore_index=True)

In [2]:
chainedOCoordsLabel=pd.read_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/chainedOCoordsLabel.csv")

In [3]:
chainedOCoordsLabel

,cell-label,0-original-x,0-original-y,15-original-x,15-original-y,30-original-x,30-original-y,45-original-x,45-original-y,60-original-x,60-original-y
0,0.0,50479.0,830.0,43299.0,8430.0,29419.0,8610.0,15459.0,9650.0,2019.0,9510.0
1,0.0,48279.0,1530.0,45499.0,7470.0,31639.0,8050.0,17739.0,8970.0,4299.0,8890.0
2,0.0,52079.0,3850.0,41719.0,5110.0,28079.0,5450.0,14299.0,6330.0,599.0,6570.0
3,0.0,52099.0,3850.0,41699.0,5110.0,28059.0,5450.0,14279.0,6350.0,599.0,6590.0
4,0.0,52119.0,3850.0,41679.0,5090.0,28039.0,5430.0,14219.0,6270.0,559.0,6550.0
...,...,...,...,...,...,...,...,...,...,...,...
40557,19.0,49319.0,3250.0,44519.0,5610.0,30639.0,6070.0,16779.0,6950.0,3359.0,6830.0
40558,19.0,49739.0,1170.0,44079.0,7970.0,31659.0,8050.0,17759.0,9010.0,4359.0,8890.0
40559,19.0,49759.0,1170.0,44079.0,7990.0,31659.0,8070.0,17779.0,9030.0,4379.0,8910.0
40560,19.0,50039.0,350.0,44059.0,8690.0,30739.0,8830.0,17039.0,9990.0,3639.0,9910.0


In [102]:
chainedOCoordsLabel = chainedOCoordsLabel.dropna()

In [104]:
chainedOCoordsLabel.to_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/chainedOCoordsLabel.csv",index=False)

In [105]:
chainedOCoordsLabel.columns

Index(['cell-label', '0-original-x', '0-original-y', '15-original-x',
       '15-original-y', '30-original-x', '30-original-y', '45-original-x',
       '45-original-y', '60-original-x', '60-original-y'],
      dtype='object')

In [106]:
adata = ad.read_h5ad("/p2/zulab/jtian/data/SA/05_CAST/output_cast9/demo1_cast9.h5ad") 

In [107]:
adata.obs

,x,y,sample
SpotIndex,,,
1,49939,5370,0
2,49959,5370,0
3,49979,5370,0
4,49999,5370,0
5,50019,5370,0
...,...,...,...
247357,3119,4970,60
247358,3139,4970,60
247359,3159,4970,60


In [108]:
obs = adata.obs[['x', 'y']].copy()
obs = obs.reset_index().rename(columns={'SpotIndex': 'obs_index'})

obs['x'] = obs['x'].astype('Int64')
obs['y'] = obs['y'].astype('Int64')

lookup = obs.set_index(['x', 'y'])['obs_index']

samples = [0, 15, 30, 45, 60]

for s in samples:
    sx = f'{s}-original-x'
    sy = f'{s}-original-y'
    out_col = f'{s}-index'

    xk = chainedOCoordsLabel[sx].astype('Int64')
    yk = chainedOCoordsLabel[sy].astype('Int64')

    mi = pd.MultiIndex.from_arrays([xk, yk], names=['x', 'y'])
    chainedOCoordsLabel[out_col] = lookup.reindex(mi).to_numpy()

In [109]:
chainedOCoordsLabel

,cell-label,0-original-x,0-original-y,15-original-x,15-original-y,30-original-x,30-original-y,45-original-x,45-original-y,60-original-x,60-original-y,0-index,15-index,30-index,45-index,60-index
0,0.0,50479.0,830.0,43299.0,8430.0,29419.0,8610.0,15459.0,9650.0,2019.0,9510.0,42345,54453,104548,152418,203457
1,0.0,48279.0,1530.0,45499.0,7470.0,31639.0,8050.0,17739.0,8970.0,4299.0,8890.0,35453,64148,110173,159394,209770
2,0.0,52079.0,3850.0,41719.0,5110.0,28079.0,5450.0,14299.0,6330.0,599.0,6570.0,10985,89234,137471,187696,235139
3,0.0,52099.0,3850.0,41699.0,5110.0,28059.0,5450.0,14279.0,6350.0,599.0,6590.0,10986,89233,137470,187501,234935
4,0.0,52119.0,3850.0,41679.0,5090.0,28039.0,5430.0,14219.0,6270.0,559.0,6550.0,10987,89427,137656,188272,235341
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47390,19.0,49319.0,3250.0,44519.0,5610.0,30639.0,6070.0,16779.0,6950.0,3359.0,6830.0,16948,84300,131464,181528,232582
47391,19.0,49739.0,1170.0,44079.0,7970.0,31659.0,8050.0,17759.0,9010.0,4359.0,8890.0,39188,58842,110174,158973,209773
47392,19.0,49759.0,1170.0,44079.0,7990.0,31659.0,8070.0,17779.0,9030.0,4379.0,8910.0,39189,58642,109966,158763,209563
47393,19.0,50039.0,350.0,44059.0,8690.0,30739.0,8830.0,17039.0,9990.0,3639.0,9910.0,45920,52294,102808,149568,200131


In [110]:
chainedIndexLabel = chainedOCoordsLabel[['cell-label','0-index','15-index','30-index','45-index','60-index']]

In [111]:
chainedIndexLabel

,cell-label,0-index,15-index,30-index,45-index,60-index
0,0.0,42345,54453,104548,152418,203457
1,0.0,35453,64148,110173,159394,209770
2,0.0,10985,89234,137471,187696,235139
3,0.0,10986,89233,137470,187501,234935
4,0.0,10987,89427,137656,188272,235341
...,...,...,...,...,...,...
47390,19.0,16948,84300,131464,181528,232582
47391,19.0,39188,58842,110174,158973,209773
47392,19.0,39189,58642,109966,158763,209563
47393,19.0,45920,52294,102808,149568,200131


In [112]:
chainedIndexLabel = chainedIndexLabel.astype(int)

In [ ]:
chainedIndexLabel.to_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/chainedIndexLabel.csv")

In [114]:
chainedIndexLabel

,cell-label,0-index,15-index,30-index,45-index,60-index
0,0,42345,54453,104548,152418,203457
1,0,35453,64148,110173,159394,209770
2,0,10985,89234,137471,187696,235139
3,0,10986,89233,137470,187501,234935
4,0,10987,89427,137656,188272,235341
...,...,...,...,...,...,...
47390,19,16948,84300,131464,181528,232582
47391,19,39188,58842,110174,158973,209773
47392,19,39189,58642,109966,158763,209563
47393,19,45920,52294,102808,149568,200131


In [115]:
coordsLabelIntensity = pd.read_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_calculateByClusterLR/coordsLabelIntensity.csv")

In [116]:
coordsLabelIntensity

,Unnamed: 0,sample,cell_label,57.0346,58.0296,59.0139,67.0191,71.0137,71.0503,72.0091,...,856.5052,859.5297,860.536,863.5618,864.5684,882.5853,921.5991,923.615,985.6046,986.6105
0,1,0,5,0.208880,0.254289,0.499496,0.399597,1.607469,0.127144,0.962665,...,0.363270,0.481332,0.163471,0.000000,0.435924,0.308779,0.000000,0.000000,0.163471,0.000000
1,2,0,5,0.000000,0.223221,0.480784,0.257563,2.481191,0.000000,1.210547,...,0.712591,0.154538,0.283319,0.394930,0.463614,0.257563,0.154538,0.000000,0.120196,0.412101
2,3,0,5,0.000000,0.000000,1.175326,0.720719,0.698543,0.000000,0.155232,...,0.177408,0.155232,0.221760,0.410255,0.000000,0.000000,0.000000,0.232848,0.654191,0.343727
3,4,0,5,0.152048,0.000000,0.557510,0.000000,0.726453,0.000000,0.278755,...,0.084471,0.000000,0.194284,0.304097,0.464592,0.312544,0.278755,0.000000,0.000000,0.000000
4,5,0,5,0.000000,0.000000,0.725190,0.382195,2.587164,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.088199,0.244997,0.244997,0.195997,0.000000,0.391995,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247356,247357,60,5,2.105920,0.399399,10.329899,2.178538,39.013979,1.216350,4.302612,...,4.393384,5.627889,0.000000,3.558278,1.670212,2.723172,1.379741,1.760985,3.739823,2.360082
247357,247358,60,5,2.115093,0.459090,16.691198,1.278893,49.745672,2.672559,4.115413,...,3.148045,4.640087,0.000000,3.672719,2.623371,1.869152,1.721587,0.950972,4.230186,2.229865
247358,247359,60,5,1.906487,2.017329,14.121301,2.793224,41.831861,0.532043,6.916556,...,3.591289,5.918976,5.054406,4.655374,2.083834,1.950823,2.394192,1.773476,5.409101,2.726719
247359,247360,60,5,1.522979,3.635498,7.148176,0.000000,30.975429,1.007131,2.554675,...,8.622027,11.299522,7.393818,7.614895,4.102218,6.091916,2.309033,1.940570,5.035657,2.972266


In [117]:
coordsLabelIntensity = coordsLabelIntensity.rename(columns={coordsLabelIntensity.columns[0]: 'SpotIndex'})


In [118]:
spotIndexIntensity = coordsLabelIntensity.drop(coordsLabelIntensity.columns[[1, 2]], axis=1)

In [119]:
spotIndexIntensity

,SpotIndex,57.0346,58.0296,59.0139,67.0191,71.0137,71.0503,72.0091,72.017,73.0294,...,856.5052,859.5297,860.536,863.5618,864.5684,882.5853,921.5991,923.615,985.6046,986.6105
0,1,0.208880,0.254289,0.499496,0.399597,1.607469,0.127144,0.962665,0.354188,0.563068,...,0.363270,0.481332,0.163471,0.000000,0.435924,0.308779,0.000000,0.000000,0.163471,0.000000
1,2,0.000000,0.223221,0.480784,0.257563,2.481191,0.000000,1.210547,0.000000,0.875714,...,0.712591,0.154538,0.283319,0.394930,0.463614,0.257563,0.154538,0.000000,0.120196,0.412101
2,3,0.000000,0.000000,1.175326,0.720719,0.698543,0.000000,0.155232,0.000000,0.000000,...,0.177408,0.155232,0.221760,0.410255,0.000000,0.000000,0.000000,0.232848,0.654191,0.343727
3,4,0.152048,0.000000,0.557510,0.000000,0.726453,0.000000,0.278755,0.143601,0.574405,...,0.084471,0.000000,0.194284,0.304097,0.464592,0.312544,0.278755,0.000000,0.000000,0.000000
4,5,0.000000,0.000000,0.725190,0.382195,2.587164,0.000000,0.000000,0.000000,1.460180,...,0.000000,0.000000,0.000000,0.088199,0.244997,0.244997,0.195997,0.000000,0.391995,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247356,247357,2.105920,0.399399,10.329899,2.178538,39.013979,1.216350,4.302612,0.000000,8.641533,...,4.393384,5.627889,0.000000,3.558278,1.670212,2.723172,1.379741,1.760985,3.739823,2.360082
247357,247358,2.115093,0.459090,16.691198,1.278893,49.745672,2.672559,4.115413,0.000000,20.085184,...,3.148045,4.640087,0.000000,3.672719,2.623371,1.869152,1.721587,0.950972,4.230186,2.229865
247358,247359,1.906487,2.017329,14.121301,2.793224,41.831861,0.532043,6.916556,0.000000,24.584808,...,3.591289,5.918976,5.054406,4.655374,2.083834,1.950823,2.394192,1.773476,5.409101,2.726719
247359,247360,1.522979,3.635498,7.148176,0.000000,30.975429,1.007131,2.554675,0.000000,14.713943,...,8.622027,11.299522,7.393818,7.614895,4.102218,6.091916,2.309033,1.940570,5.035657,2.972266


In [120]:
intensity_dict = spotIndexIntensity.set_index('SpotIndex').to_dict('index')
intensity_mz = spotIndexIntensity.columns[1:]

for index, row in chainedIndexLabel.iterrows():
    for i in range(1,6):
        SpotInd = row[i]
        if SpotInd in intensity_dict:
            for col_name in intensity_mz:
                chainedIndexLabel.at[
                    index,
                    f'{col_name}_intensity_{i+1}'
                ] = intensity_dict[SpotInd].get(col_name, None)

print(chainedIndexLabel)

       cell-label  0-index  15-index  30-index  45-index  60-index  \
0               0    42345     54453    104548    152418    203457   
1               0    35453     64148    110173    159394    209770   
2               0    10985     89234    137471    187696    235139   
3               0    10986     89233    137470    187501    234935   
4               0    10987     89427    137656    188272    235341   
...           ...      ...       ...       ...       ...       ...   
47390          19    16948     84300    131464    181528    232582   
47391          19    39188     58842    110174    158973    209773   
47392          19    39189     58642    109966    158763    209563   
47393          19    45920     52294    102808    149568    200131   
47394          19    47395     47525    100969    146346    197126   

       57.0346_intensity_2  58.0296_intensity_2  59.0139_intensity_2  \
0                 1.193864             2.956235             6.509403   
1              

In [121]:
chainedIndexLabel.to_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/chainedLabelIndexIntensity.csv",index=False)

In [122]:
chainedLabelIntensity = chainedIndexLabel.iloc[:, [0] + list(range(6, chainedIndexLabel.shape[1]))]


In [123]:
chainedLabelIntensity

,cell-label,57.0346_intensity_2,58.0296_intensity_2,59.0139_intensity_2,67.0191_intensity_2,71.0137_intensity_2,71.0503_intensity_2,72.0091_intensity_2,72.017_intensity_2,73.0294_intensity_2,...,856.5052_intensity_6,859.5297_intensity_6,860.536_intensity_6,863.5618_intensity_6,864.5684_intensity_6,882.5853_intensity_6,921.5991_intensity_6,923.615_intensity_6,985.6046_intensity_6,986.6105_intensity_6
0,0,1.193864,2.956235,6.509403,1.222290,24.019412,0.000000,1.193864,2.700407,11.114308,...,5.522259,8.695136,4.044813,6.418415,4.698765,1.404785,3.463522,1.889194,1.937635,2.664248
1,0,1.678438,1.715327,6.935086,0.590220,15.327278,1.014441,0.000000,0.885330,5.699313,...,7.691580,8.538958,6.811611,3.356919,4.073930,0.000000,2.281401,1.596981,1.564389,1.629572
2,0,0.665334,1.376554,4.749111,0.504736,19.570007,0.000000,0.848875,0.665334,5.047364,...,5.874731,6.238653,4.652995,5.016917,3.743192,2.443472,3.535237,2.079551,2.313500,2.235517
3,0,0.000000,0.402024,3.690006,0.402024,13.338582,0.258444,1.048134,0.000000,3.015180,...,6.115760,7.507176,0.000000,4.044815,0.000000,1.520850,1.132548,3.365286,1.553209,1.812077
4,0,0.000000,2.013954,3.365423,1.006977,10.599756,0.000000,1.669462,0.000000,5.405876,...,0.000000,8.401914,4.828373,0.000000,4.746536,1.473063,0.000000,3.000684,2.236873,1.773131
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47390,19,1.143931,1.085268,6.159627,4.311739,20.180112,0.000000,3.255803,0.674626,6.746258,...,5.603624,7.170514,6.294118,6.241003,2.469844,2.602631,0.000000,1.991809,2.018367,0.796724
47391,19,2.073597,1.334729,3.289153,0.000000,17.113130,0.000000,2.597954,0.000000,5.910942,...,3.402159,6.549156,6.527893,4.741759,2.211403,2.381511,1.934978,3.699848,2.742991,2.126350
47392,19,0.679172,2.595406,6.524900,0.000000,22.097338,0.751940,2.086028,1.043014,6.961511,...,5.304477,7.456092,6.859605,4.409746,4.132805,3.174165,0.383456,0.894731,3.493712,1.981190
47393,19,0.626969,0.773261,2.716864,1.734613,20.773557,0.000000,0.731463,1.253937,6.436877,...,3.311746,6.564704,4.271957,5.643686,2.331940,0.000000,1.430518,0.391923,1.587287,1.293345


In [124]:
new_cols = []

for col in chainedLabelIntensity.columns:
    if "_intensity_" in col:
        prefix, num = col.rsplit("_", 1)
        new_cols.append(f"{prefix}_{int(num)-1}")
    else:
        new_cols.append(col)

chainedLabelIntensity.columns = new_cols

In [125]:
chainedLabelIntensity

,cell-label,57.0346_intensity_1,58.0296_intensity_1,59.0139_intensity_1,67.0191_intensity_1,71.0137_intensity_1,71.0503_intensity_1,72.0091_intensity_1,72.017_intensity_1,73.0294_intensity_1,...,856.5052_intensity_5,859.5297_intensity_5,860.536_intensity_5,863.5618_intensity_5,864.5684_intensity_5,882.5853_intensity_5,921.5991_intensity_5,923.615_intensity_5,985.6046_intensity_5,986.6105_intensity_5
0,0,1.193864,2.956235,6.509403,1.222290,24.019412,0.000000,1.193864,2.700407,11.114308,...,5.522259,8.695136,4.044813,6.418415,4.698765,1.404785,3.463522,1.889194,1.937635,2.664248
1,0,1.678438,1.715327,6.935086,0.590220,15.327278,1.014441,0.000000,0.885330,5.699313,...,7.691580,8.538958,6.811611,3.356919,4.073930,0.000000,2.281401,1.596981,1.564389,1.629572
2,0,0.665334,1.376554,4.749111,0.504736,19.570007,0.000000,0.848875,0.665334,5.047364,...,5.874731,6.238653,4.652995,5.016917,3.743192,2.443472,3.535237,2.079551,2.313500,2.235517
3,0,0.000000,0.402024,3.690006,0.402024,13.338582,0.258444,1.048134,0.000000,3.015180,...,6.115760,7.507176,0.000000,4.044815,0.000000,1.520850,1.132548,3.365286,1.553209,1.812077
4,0,0.000000,2.013954,3.365423,1.006977,10.599756,0.000000,1.669462,0.000000,5.405876,...,0.000000,8.401914,4.828373,0.000000,4.746536,1.473063,0.000000,3.000684,2.236873,1.773131
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47390,19,1.143931,1.085268,6.159627,4.311739,20.180112,0.000000,3.255803,0.674626,6.746258,...,5.603624,7.170514,6.294118,6.241003,2.469844,2.602631,0.000000,1.991809,2.018367,0.796724
47391,19,2.073597,1.334729,3.289153,0.000000,17.113130,0.000000,2.597954,0.000000,5.910942,...,3.402159,6.549156,6.527893,4.741759,2.211403,2.381511,1.934978,3.699848,2.742991,2.126350
47392,19,0.679172,2.595406,6.524900,0.000000,22.097338,0.751940,2.086028,1.043014,6.961511,...,5.304477,7.456092,6.859605,4.409746,4.132805,3.174165,0.383456,0.894731,3.493712,1.981190
47393,19,0.626969,0.773261,2.716864,1.734613,20.773557,0.000000,0.731463,1.253937,6.436877,...,3.311746,6.564704,4.271957,5.643686,2.331940,0.000000,1.430518,0.391923,1.587287,1.293345


In [127]:
chainedIndexLabel = pd.read_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/chainedLabelIndexIntensity.csv")

In [128]:
chainedIndexLabel

,cell-label,0-index,15-index,30-index,45-index,60-index,57.0346_intensity_2,58.0296_intensity_2,59.0139_intensity_2,67.0191_intensity_2,...,856.5052_intensity_6,859.5297_intensity_6,860.536_intensity_6,863.5618_intensity_6,864.5684_intensity_6,882.5853_intensity_6,921.5991_intensity_6,923.615_intensity_6,985.6046_intensity_6,986.6105_intensity_6
0,0,42345,54453,104548,152418,203457,1.193864,2.956235,6.509403,1.222290,...,5.522259,8.695136,4.044813,6.418415,4.698765,1.404785,3.463522,1.889194,1.937635,2.664248
1,0,35453,64148,110173,159394,209770,1.678438,1.715327,6.935086,0.590220,...,7.691580,8.538958,6.811611,3.356919,4.073930,0.000000,2.281401,1.596981,1.564389,1.629572
2,0,10985,89234,137471,187696,235139,0.665334,1.376554,4.749111,0.504736,...,5.874731,6.238653,4.652995,5.016917,3.743192,2.443472,3.535237,2.079551,2.313500,2.235517
3,0,10986,89233,137470,187501,234935,0.000000,0.402024,3.690006,0.402024,...,6.115760,7.507176,0.000000,4.044815,0.000000,1.520850,1.132548,3.365286,1.553209,1.812077
4,0,10987,89427,137656,188272,235341,0.000000,2.013954,3.365423,1.006977,...,0.000000,8.401914,4.828373,0.000000,4.746536,1.473063,0.000000,3.000684,2.236873,1.773131
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40557,19,16948,84300,131464,181528,232582,1.143931,1.085268,6.159627,4.311739,...,5.603624,7.170514,6.294118,6.241003,2.469844,2.602631,0.000000,1.991809,2.018367,0.796724
40558,19,39188,58842,110174,158973,209773,2.073597,1.334729,3.289153,0.000000,...,3.402159,6.549156,6.527893,4.741759,2.211403,2.381511,1.934978,3.699848,2.742991,2.126350
40559,19,39189,58642,109966,158763,209563,0.679172,2.595406,6.524900,0.000000,...,5.304477,7.456092,6.859605,4.409746,4.132805,3.174165,0.383456,0.894731,3.493712,1.981190
40560,19,45920,52294,102808,149568,200131,0.626969,0.773261,2.716864,1.734613,...,3.311746,6.564704,4.271957,5.643686,2.331940,0.000000,1.430518,0.391923,1.587287,1.293345


In [129]:
new_cols = []

for col in chainedIndexLabel.columns:
    if "_intensity_" in col:
        prefix, num = col.rsplit("_", 1)
        new_cols.append(f"{prefix}_{int(num)-1}")
    else:
        new_cols.append(col)

chainedIndexLabel.columns = new_cols

In [130]:
chainedIndexLabel

,cell-label,0-index,15-index,30-index,45-index,60-index,57.0346_intensity_1,58.0296_intensity_1,59.0139_intensity_1,67.0191_intensity_1,...,856.5052_intensity_5,859.5297_intensity_5,860.536_intensity_5,863.5618_intensity_5,864.5684_intensity_5,882.5853_intensity_5,921.5991_intensity_5,923.615_intensity_5,985.6046_intensity_5,986.6105_intensity_5
0,0,42345,54453,104548,152418,203457,1.193864,2.956235,6.509403,1.222290,...,5.522259,8.695136,4.044813,6.418415,4.698765,1.404785,3.463522,1.889194,1.937635,2.664248
1,0,35453,64148,110173,159394,209770,1.678438,1.715327,6.935086,0.590220,...,7.691580,8.538958,6.811611,3.356919,4.073930,0.000000,2.281401,1.596981,1.564389,1.629572
2,0,10985,89234,137471,187696,235139,0.665334,1.376554,4.749111,0.504736,...,5.874731,6.238653,4.652995,5.016917,3.743192,2.443472,3.535237,2.079551,2.313500,2.235517
3,0,10986,89233,137470,187501,234935,0.000000,0.402024,3.690006,0.402024,...,6.115760,7.507176,0.000000,4.044815,0.000000,1.520850,1.132548,3.365286,1.553209,1.812077
4,0,10987,89427,137656,188272,235341,0.000000,2.013954,3.365423,1.006977,...,0.000000,8.401914,4.828373,0.000000,4.746536,1.473063,0.000000,3.000684,2.236873,1.773131
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40557,19,16948,84300,131464,181528,232582,1.143931,1.085268,6.159627,4.311739,...,5.603624,7.170514,6.294118,6.241003,2.469844,2.602631,0.000000,1.991809,2.018367,0.796724
40558,19,39188,58842,110174,158973,209773,2.073597,1.334729,3.289153,0.000000,...,3.402159,6.549156,6.527893,4.741759,2.211403,2.381511,1.934978,3.699848,2.742991,2.126350
40559,19,39189,58642,109966,158763,209563,0.679172,2.595406,6.524900,0.000000,...,5.304477,7.456092,6.859605,4.409746,4.132805,3.174165,0.383456,0.894731,3.493712,1.981190
40560,19,45920,52294,102808,149568,200131,0.626969,0.773261,2.716864,1.734613,...,3.311746,6.564704,4.271957,5.643686,2.331940,0.000000,1.430518,0.391923,1.587287,1.293345


In [132]:
chainedIndexLabel.to_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/chainedLabelIndexIntensity.csv",index=False)

In [133]:
chainedIndexLabel = pd.read_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/chainedLabelIndexIntensity.csv")

In [134]:
chainedIntensity = chainedIndexLabel.iloc[:,6:]

In [135]:
chainedIntensity

,57.0346_intensity_1,58.0296_intensity_1,59.0139_intensity_1,67.0191_intensity_1,71.0137_intensity_1,71.0503_intensity_1,72.0091_intensity_1,72.017_intensity_1,73.0294_intensity_1,74.0246_intensity_1,...,856.5052_intensity_5,859.5297_intensity_5,860.536_intensity_5,863.5618_intensity_5,864.5684_intensity_5,882.5853_intensity_5,921.5991_intensity_5,923.615_intensity_5,985.6046_intensity_5,986.6105_intensity_5
0,1.193864,2.956235,6.509403,1.222290,24.019412,0.000000,1.193864,2.700407,11.114308,6.282000,...,5.522259,8.695136,4.044813,6.418415,4.698765,1.404785,3.463522,1.889194,1.937635,2.664248
1,1.678438,1.715327,6.935086,0.590220,15.327278,1.014441,0.000000,0.885330,5.699313,2.914212,...,7.691580,8.538958,6.811611,3.356919,4.073930,0.000000,2.281401,1.596981,1.564389,1.629572
2,0.665334,1.376554,4.749111,0.504736,19.570007,0.000000,0.848875,0.665334,5.047364,1.766577,...,5.874731,6.238653,4.652995,5.016917,3.743192,2.443472,3.535237,2.079551,2.313500,2.235517
3,0.000000,0.402024,3.690006,0.402024,13.338582,0.258444,1.048134,0.000000,3.015180,1.995762,...,6.115760,7.507176,0.000000,4.044815,0.000000,1.520850,1.132548,3.365286,1.553209,1.812077
4,0.000000,2.013954,3.365423,1.006977,10.599756,0.000000,1.669462,0.000000,5.405876,2.702938,...,0.000000,8.401914,4.828373,0.000000,4.746536,1.473063,0.000000,3.000684,2.236873,1.773131
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40557,1.143931,1.085268,6.159627,4.311739,20.180112,0.000000,3.255803,0.674626,6.746258,7.919521,...,5.603624,7.170514,6.294118,6.241003,2.469844,2.602631,0.000000,1.991809,2.018367,0.796724
40558,2.073597,1.334729,3.289153,0.000000,17.113130,0.000000,2.597954,0.000000,5.910942,6.673644,...,3.402159,6.549156,6.527893,4.741759,2.211403,2.381511,1.934978,3.699848,2.742991,2.126350
40559,0.679172,2.595406,6.524900,0.000000,22.097338,0.751940,2.086028,1.043014,6.961511,5.797216,...,5.304477,7.456092,6.859605,4.409746,4.132805,3.174165,0.383456,0.894731,3.493712,1.981190
40560,0.626969,0.773261,2.716864,1.734613,20.773557,0.000000,0.731463,1.253937,6.436877,4.012598,...,3.311746,6.564704,4.271957,5.643686,2.331940,0.000000,1.430518,0.391923,1.587287,1.293345


In [136]:
chainedIntensity = chainedIntensity.reindex(
    sorted(
        chainedIntensity.columns,
        key=lambda x: (
            float(x.split('_')[0]),
            int(x.split('_')[-1])
        )
    ),
    axis=1
)

In [137]:
chainedIntensity

,57.0346_intensity_1,57.0346_intensity_2,57.0346_intensity_3,57.0346_intensity_4,57.0346_intensity_5,58.0296_intensity_1,58.0296_intensity_2,58.0296_intensity_3,58.0296_intensity_4,58.0296_intensity_5,...,985.6046_intensity_1,985.6046_intensity_2,985.6046_intensity_3,985.6046_intensity_4,985.6046_intensity_5,986.6105_intensity_1,986.6105_intensity_2,986.6105_intensity_3,986.6105_intensity_4,986.6105_intensity_5
0,1.193864,0.949698,2.756419,0.000000,0.000000,2.956235,1.656449,0.000000,1.867907,1.259463,...,0.483231,1.170557,4.549430,0.000000,1.937635,2.188751,0.000000,2.087386,4.643083,2.664248
1,1.678438,0.877502,0.954492,0.000000,1.140700,1.715327,1.541558,1.978824,2.992746,1.988078,...,1.217329,1.280679,2.607392,1.153454,1.564389,1.125107,0.948651,0.000000,1.808117,1.629572
2,0.665334,1.293815,0.000000,0.000000,0.000000,1.376554,0.000000,0.000000,0.968186,4.601006,...,0.780047,0.843793,1.505947,1.651611,2.313500,0.848875,0.000000,1.254956,3.075413,2.235517
3,0.000000,0.000000,0.000000,0.419034,0.000000,0.402024,3.052674,0.000000,3.033005,1.585567,...,1.493232,1.290728,2.431715,2.015352,1.553209,0.000000,2.356090,1.317179,1.117423,1.812077
4,0.000000,1.056128,1.997653,0.000000,2.318710,2.013954,0.000000,0.000000,0.000000,1.336668,...,1.960955,0.000000,1.264569,1.704679,2.236873,0.000000,1.280156,1.686092,1.658607,1.773131
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40557,1.143931,0.000000,0.726758,0.000000,2.416729,1.085268,0.000000,1.877457,1.398805,4.594440,...,1.055936,1.144241,2.079334,0.000000,2.018367,0.791952,0.722678,1.917832,1.482733,0.796724
40558,2.073597,1.925809,0.000000,0.000000,1.977505,1.334729,1.719472,0.804923,1.406064,2.572883,...,2.597954,2.269703,0.916718,2.633580,2.742991,2.478782,1.169241,0.603693,3.347772,2.126350
40559,0.679172,1.507481,1.362612,0.528192,0.000000,2.595406,2.934203,1.017647,0.000000,1.448612,...,0.000000,1.534400,2.138783,3.672195,3.493712,1.673673,2.961123,1.362612,2.942787,1.981190
40560,0.626969,0.840689,0.000000,2.316380,0.333134,0.773261,0.000000,1.799405,0.000000,0.470307,...,0.376181,0.945775,3.370314,2.697901,1.587287,0.000000,0.805660,1.885091,1.008306,1.293345


In [138]:
chainedIntensity = chainedIntensity.T

In [139]:
chainedIntensity

,0,1,2,3,4,5,6,7,8,9,...,40552,40553,40554,40555,40556,40557,40558,40559,40560,40561
57.0346_intensity_1,1.193864,1.678438,0.665334,0.000000,0.000000,0.000000,0.000000,0.452662,0.732017,0.845409,...,0.591729,1.574690,0.000000,1.561257,2.261133,1.143931,2.073597,0.679172,0.626969,0.000000
57.0346_intensity_2,0.949698,0.877502,1.293815,0.000000,1.056128,1.297020,0.000000,0.829797,0.000000,0.434809,...,0.000000,1.576006,1.418399,1.349555,1.491752,0.000000,1.925809,1.507481,0.840689,1.738955
57.0346_intensity_3,2.756419,0.954492,0.000000,0.000000,1.997653,0.626225,1.315433,0.000000,0.000000,1.558544,...,1.341126,2.160151,0.000000,0.000000,0.000000,0.726758,0.000000,1.362612,0.000000,2.020036
57.0346_intensity_4,0.000000,0.000000,0.000000,0.419034,0.000000,0.000000,0.000000,2.156734,1.080977,2.007565,...,2.295319,0.000000,0.000000,0.000000,2.349440,0.000000,0.000000,0.528192,2.316380,0.000000
57.0346_intensity_5,0.000000,1.140700,0.000000,0.000000,2.318710,1.292148,0.000000,0.000000,1.258113,0.000000,...,0.000000,3.802232,3.104297,1.880730,0.000000,2.416729,1.977505,0.000000,0.333134,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
986.6105_intensity_1,2.188751,1.125107,0.848875,0.000000,0.000000,1.576276,1.527744,0.000000,1.523386,0.000000,...,1.065112,1.066725,1.269041,1.122154,0.763499,0.791952,2.478782,1.673673,0.000000,1.893729
986.6105_intensity_2,0.000000,0.948651,0.000000,2.356090,1.280156,1.033856,2.076849,0.929373,1.641007,2.010993,...,2.162195,0.000000,1.466480,1.012166,1.437507,0.722678,1.169241,2.961123,0.805660,2.205503
986.6105_intensity_3,2.087386,0.000000,1.254956,1.317179,1.686092,1.451704,0.889851,1.617012,2.864092,1.508804,...,2.752837,3.029178,1.298957,0.686334,0.000000,1.917832,0.603693,1.362612,1.885091,0.000000
986.6105_intensity_4,4.643083,1.808117,3.075413,1.117423,1.658607,0.741018,1.340030,1.983425,2.133508,2.007565,...,1.272105,1.873034,2.101682,1.912068,2.802344,1.482733,3.347772,2.942787,1.008306,0.880870


In [140]:
group_names = []

for i in range(0, len(chainedIntensity), 5):
    group_name = chainedIntensity.index[i].split('_')[0]
    group_names.append(group_name)

grouped_result = pd.DataFrame()

for i, group_name in enumerate(group_names):
    group_data = chainedIntensity.iloc[i*5:(i+1)*5]
    group_data.index = [group_name] * len(group_data)
    grouped_result = pd.concat([grouped_result, group_data])

In [141]:
grouped_result

,0,1,2,3,4,5,6,7,8,9,...,40552,40553,40554,40555,40556,40557,40558,40559,40560,40561
57.0346,1.193864,1.678438,0.665334,0.000000,0.000000,0.000000,0.000000,0.452662,0.732017,0.845409,...,0.591729,1.574690,0.000000,1.561257,2.261133,1.143931,2.073597,0.679172,0.626969,0.000000
57.0346,0.949698,0.877502,1.293815,0.000000,1.056128,1.297020,0.000000,0.829797,0.000000,0.434809,...,0.000000,1.576006,1.418399,1.349555,1.491752,0.000000,1.925809,1.507481,0.840689,1.738955
57.0346,2.756419,0.954492,0.000000,0.000000,1.997653,0.626225,1.315433,0.000000,0.000000,1.558544,...,1.341126,2.160151,0.000000,0.000000,0.000000,0.726758,0.000000,1.362612,0.000000,2.020036
57.0346,0.000000,0.000000,0.000000,0.419034,0.000000,0.000000,0.000000,2.156734,1.080977,2.007565,...,2.295319,0.000000,0.000000,0.000000,2.349440,0.000000,0.000000,0.528192,2.316380,0.000000
57.0346,0.000000,1.140700,0.000000,0.000000,2.318710,1.292148,0.000000,0.000000,1.258113,0.000000,...,0.000000,3.802232,3.104297,1.880730,0.000000,2.416729,1.977505,0.000000,0.333134,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
986.6105,2.188751,1.125107,0.848875,0.000000,0.000000,1.576276,1.527744,0.000000,1.523386,0.000000,...,1.065112,1.066725,1.269041,1.122154,0.763499,0.791952,2.478782,1.673673,0.000000,1.893729
986.6105,0.000000,0.948651,0.000000,2.356090,1.280156,1.033856,2.076849,0.929373,1.641007,2.010993,...,2.162195,0.000000,1.466480,1.012166,1.437507,0.722678,1.169241,2.961123,0.805660,2.205503
986.6105,2.087386,0.000000,1.254956,1.317179,1.686092,1.451704,0.889851,1.617012,2.864092,1.508804,...,2.752837,3.029178,1.298957,0.686334,0.000000,1.917832,0.603693,1.362612,1.885091,0.000000
986.6105,4.643083,1.808117,3.075413,1.117423,1.658607,0.741018,1.340030,1.983425,2.133508,2.007565,...,1.272105,1.873034,2.101682,1.912068,2.802344,1.482733,3.347772,2.942787,1.008306,0.880870


In [142]:
grouped_result.to_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/grouped_result.csv")

In [145]:
def linreg_stats(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    xm = x.mean()
    ym = y.mean()
    dx = x - xm
    dy = y - ym

    varx = np.sum(dx * dx)
    if varx == 0:
        return np.nan, np.nan, np.nan, np.nan

    slope = np.sum(dx * dy) / varx
    intercept = ym - slope * xm

    yhat = slope * x + intercept
    ss_res = np.sum((y - yhat) ** 2)
    ss_tot = np.sum((y - ym) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else np.nan

    # y=0 时 x 截距：x0 = -b/a；取绝对值
    x0_abs = np.abs(-intercept / slope) if slope != 0 else np.nan
    return slope, intercept, r2, x0_abs

In [146]:
# 假设 'grouped_result' 是之前的 DataFrame，按照索引列分组
x_values = np.array([0, 15, 30, 45, 60])  # x 轴

# 初始化线性回归模型
model = LinearRegression()

# 创建一个空的列表，用来存储每个组的回归结果
regression_results = []

# 按索引分组
grouped = grouped_result.groupby(grouped_result.index)

# Step 1: 对每个组进行回归计算
for group_name, group_data in grouped:
    group_results = {}  # 存储每个组的回归结果
    
    # 对组内每列数据进行回归
    for col_name in group_data.columns:
        y_values = group_data[col_name].values  # y值
        
        # 进行线性回归
        slope, intercept, r2, x0_abs=linreg_stats(x_values, y_values)
        
        # 将结果存储到字典中，列名格式为原列名加上后缀 '_*'
        group_results[f'{col_name}_Slope'] = slope
        group_results[f'{col_name}_Intercept'] = intercept
        group_results[f'{col_name}_R2'] = r2
        group_results[f'{col_name}_x_intercept_abs'] = x0_abs
    
    # 将每个组的回归结果存入列表
    regression_results.append(group_results)

# Step 2: 使用 pd.DataFrame() 将结果列表转换为 DataFrame
regression_df = pd.DataFrame(regression_results)

# 将回归结果的行名设置为原来的组名
regression_df.index = grouped_result.index.unique()

# 输出回归结果
print(regression_df)



           0_Slope  0_Intercept      0_R2  0_x_intercept_abs   1_Slope  \
57.0346   0.022244     0.438949  0.574502          19.733264  0.045240   
58.0296   0.128146    12.166875  0.230818          94.945478  0.110362   
59.0139   0.001277     0.748815  0.001614         586.561415  0.010225   
67.0191   0.011641     0.654437  0.221161          56.217361  0.046168   
71.0137   0.043132    -0.460188  0.788654          10.669318  0.000000   
...            ...          ...       ...                ...       ...   
882.5853  0.011588     1.280521  0.023604         110.500791  0.003779   
921.5991  0.037294     1.197878  0.285295          32.120001  0.012456   
923.615   0.032118    26.984932  0.098887         840.171129 -0.132195   
985.6046  0.051259     0.882702  0.386339          17.220522  0.038164   
986.6105  0.020294     3.126196  0.273316         154.044621  0.054912   

          1_Intercept      1_R2  1_x_intercept_abs   2_Slope  2_Intercept  \
57.0346      0.244048  0.705070   

In [147]:
regression_df.to_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/LRSingleCell.csv",index=True)

In [15]:
regression_df=pd.read_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/LRSingleCell.csv",index_col=0)

In [7]:
regression_df

,0_Slope,0_Intercept,0_R2,0_x_intercept_abs,1_Slope,1_Intercept,1_R2,1_x_intercept_abs,2_Slope,2_Intercept,...,40559_R2,40559_x_intercept_abs,40560_Slope,40560_Intercept,40560_R2,40560_x_intercept_abs,40561_Slope,40561_Intercept,40561_R2,40561_x_intercept_abs
57.0346,0.022244,0.438949,0.574502,19.733264,0.045240,0.244048,0.705070,5.394555,0.059337,0.154726,...,0.445710,4.200284,0.025737,0.441892,0.231255,17.169640,0.049826,0.334489,0.288477,6.713135
58.0296,0.128146,12.166875,0.230818,94.945478,0.110362,7.893238,0.793175,71.521029,0.185908,6.919793,...,0.599662,32.558628,0.096704,11.737848,0.255572,121.378554,-0.003856,12.993832,0.000481,3369.789828
59.0139,0.001277,0.748815,0.001614,586.561415,0.010225,1.126115,0.219852,110.134540,-0.001953,1.289335,...,0.180978,53.228764,0.005871,0.298423,0.086531,50.828077,0.009986,1.441252,0.032409,144.320375
67.0191,0.011641,0.654437,0.221161,56.217361,0.046168,2.103784,0.566991,45.567517,0.018325,1.244204,...,0.005946,1533.660393,0.033673,1.167339,0.196666,34.666869,0.056788,1.801027,0.636921,31.714972
71.0137,0.043132,-0.460188,0.788654,10.669318,0.000000,0.083809,0.000000,NaN,0.000000,0.000000,...,0.104367,41.954608,0.023777,-0.356650,0.500000,15.000000,-0.023674,1.677376,0.287347,70.852844
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
882.5853,0.011588,1.280521,0.023604,110.500791,0.003779,1.451270,0.022047,384.004675,0.025831,0.644035,...,0.912780,5.634511,0.027829,0.960624,0.286870,34.518925,-0.009595,2.476053,0.031467,258.049114
921.5991,0.037294,1.197878,0.285295,32.120001,0.012456,0.728610,0.173213,58.494856,0.038991,0.313213,...,0.016515,519.091115,0.018596,0.440613,0.407935,23.694531,-0.012314,1.691943,0.108448,137.397379
923.6150,0.032118,26.984932,0.098887,840.171129,-0.132195,25.094725,0.296096,189.831057,0.214465,18.338929,...,0.703000,170.636664,-0.178595,29.047359,0.326949,162.643350,0.025167,24.091048,0.043643,957.258305
985.6046,0.051259,0.882702,0.386339,17.220522,0.038164,1.523792,0.556748,39.927560,-0.004048,2.449653,...,0.800795,58.787272,-0.015950,2.462093,0.398811,154.363090,0.010874,1.453707,0.035572,133.681269


In [26]:
r2Results = regression_df.loc[:,regression_df.columns.str.contains('R2')]
r209Counts = (r2Results>=0.9).sum(axis=1)


In [27]:
r209Counts

57.0346     1492
58.0296     1991
59.0139      595
67.0191     1066
71.0137      167
            ... 
882.5853    1031
921.5991     912
923.6150     754
985.6046     686
986.6105     713
Length: 696, dtype: int64

In [28]:
(r209Counts >= 8000).sum()

27

In [29]:
r208Counts = (r2Results>=0.8).sum(axis=1)

In [30]:
r208Counts

57.0346     3947
58.0296     5162
59.0139     1768
67.0191     2972
71.0137      415
            ... 
882.5853    2790
921.5991    2454
923.6150    2167
985.6046    1926
986.6105    1991
Length: 696, dtype: int64

In [31]:
(r208Counts >= 8000).sum()

89

In [32]:
r207Counts = (r2Results>=0.7).sum(axis=1)

In [33]:
r207Counts

57.0346     7132
58.0296     8803
59.0139     3571
67.0191     5315
71.0137     2122
            ... 
882.5853    5103
921.5991    4417
923.6150    4082
985.6046    3652
986.6105    3798
Length: 696, dtype: int64

In [34]:
(r207Counts >= 8000).sum()

160

In [168]:
regression_df

,0_Slope,0_Intercept,0_R2,0_x_intercept_abs,1_Slope,1_Intercept,1_R2,1_x_intercept_abs,2_Slope,2_Intercept,...,40559_R2,40559_x_intercept_abs,40560_Slope,40560_Intercept,40560_R2,40560_x_intercept_abs,40561_Slope,40561_Intercept,40561_R2,40561_x_intercept_abs
57.0346,0.022244,0.438949,0.574502,19.733264,0.045240,0.244048,0.705070,5.394555,0.059337,0.154726,...,0.445710,4.200284,0.025737,0.441892,0.231255,17.169640,0.049826,0.334489,0.288477,6.713135
58.0296,0.128146,12.166875,0.230818,94.945478,0.110362,7.893238,0.793175,71.521029,0.185908,6.919793,...,0.599662,32.558628,0.096704,11.737848,0.255572,121.378554,-0.003856,12.993832,0.000481,3369.789828
59.0139,0.001277,0.748815,0.001614,586.561415,0.010225,1.126115,0.219852,110.134540,-0.001953,1.289335,...,0.180978,53.228764,0.005871,0.298423,0.086531,50.828077,0.009986,1.441252,0.032409,144.320375
67.0191,0.011641,0.654437,0.221161,56.217361,0.046168,2.103784,0.566991,45.567517,0.018325,1.244204,...,0.005946,1533.660393,0.033673,1.167339,0.196666,34.666869,0.056788,1.801027,0.636921,31.714972
71.0137,0.043132,-0.460188,0.788654,10.669318,0.000000,0.083809,0.000000,NaN,0.000000,0.000000,...,0.104367,41.954608,0.023777,-0.356650,0.500000,15.000000,-0.023674,1.677376,0.287347,70.852844
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
882.5853,0.011588,1.280521,0.023604,110.500791,0.003779,1.451270,0.022047,384.004675,0.025831,0.644035,...,0.912780,5.634511,0.027829,0.960624,0.286870,34.518925,-0.009595,2.476053,0.031467,258.049114
921.5991,0.037294,1.197878,0.285295,32.120001,0.012456,0.728610,0.173213,58.494856,0.038991,0.313213,...,0.016515,519.091115,0.018596,0.440613,0.407935,23.694531,-0.012314,1.691943,0.108448,137.397379
923.6150,0.032118,26.984932,0.098887,840.171129,-0.132195,25.094725,0.296096,189.831057,0.214465,18.338929,...,0.703000,170.636664,-0.178595,29.047359,0.326949,162.643350,0.025167,24.091048,0.043643,957.258305
985.6046,0.051259,0.882702,0.386339,17.220522,0.038164,1.523792,0.556748,39.927560,-0.004048,2.449653,...,0.800795,58.787272,-0.015950,2.462093,0.398811,154.363090,0.010874,1.453707,0.035572,133.681269


In [17]:
abs_df = regression_df.loc[:,regression_df.columns.str.contains('R2|abs')]

In [11]:
abs_df

,0_R2,0_x_intercept_abs,1_R2,1_x_intercept_abs,2_R2,2_x_intercept_abs,3_R2,3_x_intercept_abs,4_R2,4_x_intercept_abs,...,40557_R2,40557_x_intercept_abs,40558_R2,40558_x_intercept_abs,40559_R2,40559_x_intercept_abs,40560_R2,40560_x_intercept_abs,40561_R2,40561_x_intercept_abs
57.0346,0.574502,19.733264,0.705070,5.394555,0.827995,2.607599,0.038342,117.489639,0.430325,31.677846,...,0.299041,40.340012,0.387592,1.343736e+00,0.445710,4.200284,0.231255,17.169640,0.288477,6.713135
58.0296,0.230818,94.945478,0.793175,71.521029,0.786438,37.221524,0.669062,46.878295,0.824484,62.844429,...,0.819167,42.243887,0.536842,1.611953e+00,0.599662,32.558628,0.255572,121.378554,0.000481,3369.789828
59.0139,0.001614,586.561415,0.219852,110.134540,0.019192,660.255782,0.032348,171.744356,0.003241,403.268594,...,0.003974,354.502873,0.413113,2.043795e+01,0.180978,53.228764,0.086531,50.828077,0.032409,144.320375
67.0191,0.221161,56.217361,0.566991,45.567517,0.156105,67.895470,0.526616,1.822120,0.056629,205.113061,...,0.056949,178.487743,0.702424,1.614285e+01,0.005946,1533.660393,0.196666,34.666869,0.636921,31.714972
71.0137,0.788654,10.669318,0.000000,NaN,NaN,NaN,0.012010,118.844426,0.500000,45.000000,...,0.125000,60.000000,0.000000,6.457125e+17,0.104367,41.954608,0.500000,15.000000,0.287347,70.852844
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
882.5853,0.023604,110.500791,0.022047,384.004675,0.937802,24.932144,0.083870,282.022159,0.165287,65.285084,...,0.021092,212.033471,0.018694,4.820087e+02,0.912780,5.634511,0.286870,34.518925,0.031467,258.049114
921.5991,0.285295,32.120001,0.173213,58.494856,0.594344,8.032881,0.183621,53.036760,0.703405,18.905373,...,0.052156,192.658473,0.046339,1.679927e+02,0.016515,519.091115,0.407935,23.694531,0.108448,137.397379
923.6150,0.098887,840.171129,0.296096,189.831057,0.788906,85.510282,0.069835,290.418157,0.741877,64.164312,...,0.787495,92.876754,0.049662,5.444558e+02,0.703000,170.636664,0.326949,162.643350,0.043643,957.258305
985.6046,0.386339,17.220522,0.556748,39.927560,0.022247,605.175796,0.353356,54.903227,0.556455,134.748856,...,0.136449,114.153880,0.904564,3.299666e+00,0.800795,58.787272,0.398811,154.363090,0.035572,133.681269


In [20]:
idx = r207Counts[r207Counts >= 8000].index

abs_df_filtered = abs_df.copy()

for i in range(0, abs_df_filtered.shape[1], 2):
    group_cols = abs_df_filtered.columns[i:i+2]

    # 这一组里分别找到 R2 列和 abs 列
    r2_col = [col for col in group_cols if 'R2' in col][0]
    abs_col = [col for col in group_cols if 'abs' in col][0]

    # R2 < 0.7 的位置，把对应 abs 值设为 NaN
    abs_df_filtered.loc[abs_df_filtered[r2_col] < 0.7, abs_col] = np.nan

In [192]:
r207Spots8000=r207Counts[r207Counts >= 8000]
r207Spots8000.to_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/r207Spots8000.csv",index=True)

In [21]:
abs_df_filtered=abs_df_filtered.loc[idx,:]

In [14]:
abs_df_filtered

,0_R2,0_x_intercept_abs,1_R2,1_x_intercept_abs,2_R2,2_x_intercept_abs,3_R2,3_x_intercept_abs,4_R2,4_x_intercept_abs,...,40557_R2,40557_x_intercept_abs,40558_R2,40558_x_intercept_abs,40559_R2,40559_x_intercept_abs,40560_R2,40560_x_intercept_abs,40561_R2,40561_x_intercept_abs
58.0296,0.230818,NaN,0.793175,71.521029,0.786438,37.221524,0.669062,NaN,0.824484,62.844429,...,0.819167,42.243887,0.536842,NaN,0.599662,NaN,0.255572,NaN,0.000481,NaN
71.0503,0.116835,NaN,0.471518,NaN,0.408221,NaN,0.832325,37.664302,0.595771,NaN,...,0.413860,NaN,0.828387,29.431502,0.814243,29.390637,0.390209,NaN,0.445643,NaN
74.0246,0.336096,NaN,0.661421,NaN,0.813940,22.491404,0.901802,18.797747,0.769549,23.308197,...,0.901280,23.491930,0.799318,17.432832,0.893133,7.647542,0.600840,NaN,0.712564,6.369371
84.0454,0.600388,NaN,0.876338,31.004770,0.955498,59.076727,0.574644,NaN,0.115896,NaN,...,0.200500,NaN,0.644171,NaN,0.239225,NaN,0.803259,12.681492,0.573912,NaN
99.0077,0.232241,NaN,0.127425,NaN,0.178441,NaN,0.788924,43.470226,0.544956,NaN,...,0.118136,NaN,0.234119,NaN,0.235440,NaN,0.036387,NaN,0.022630,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
784.5070,0.648066,NaN,0.275622,NaN,0.276299,NaN,0.473883,NaN,0.313785,NaN,...,0.053820,NaN,0.327691,NaN,0.426328,NaN,0.600879,NaN,0.185353,NaN
818.4730,0.921458,11.649830,0.803743,14.406159,0.800231,10.836148,0.912023,10.817250,0.867826,37.271538,...,0.854217,9.082285,0.654534,NaN,0.868717,1.649238,0.712504,15.104056,0.648631,NaN
831.4993,0.322098,NaN,0.103124,NaN,0.553121,NaN,0.150556,NaN,0.852234,21.730873,...,0.853926,19.464816,0.712625,0.094391,0.606693,NaN,0.236273,NaN,0.579470,NaN
856.5052,0.080865,NaN,0.426207,NaN,0.316985,NaN,0.208134,NaN,0.353046,NaN,...,0.206983,NaN,0.853729,2.248061,0.115061,NaN,0.149415,NaN,0.805538,1.851916


In [22]:
abs_df = abs_df_filtered.loc[:, abs_df_filtered.columns.str.contains('abs')]

In [23]:
abs_df = abs_df.T

In [17]:
abs_df

,58.0296,71.0503,74.0246,84.0454,99.0077,101.0239,102.0275,114.9259,115.0035,118.0505,...,757.4864,758.5665,775.4865,781.4886,783.5035,784.5070,818.4730,831.4993,856.5052,860.5360
0_x_intercept_abs,NaN,NaN,NaN,NaN,NaN,NaN,54.500993,33.153667,29.568411,NaN,...,NaN,NaN,3.443140,NaN,40.829728,NaN,11.649830,NaN,NaN,113.407464
1_x_intercept_abs,71.521029,NaN,NaN,31.004770,NaN,NaN,32.604357,NaN,7.987146,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,14.406159,NaN,NaN,107.342227
2_x_intercept_abs,37.221524,NaN,22.491404,59.076727,NaN,88.804098,16.879574,NaN,NaN,NaN,...,2.300068,NaN,NaN,NaN,50.001698,NaN,10.836148,NaN,NaN,73.631896
3_x_intercept_abs,NaN,37.664302,18.797747,NaN,43.470226,NaN,NaN,19.012033,5.225182,29.492769,...,NaN,NaN,NaN,22.946539,9.948982,NaN,10.817250,NaN,NaN,41.851645
4_x_intercept_abs,62.844429,NaN,23.308197,NaN,NaN,NaN,28.590537,12.322394,9.863505,NaN,...,12.226012,0.463222,NaN,NaN,NaN,NaN,37.271538,21.730873,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40557_x_intercept_abs,42.243887,NaN,23.491930,NaN,NaN,64.634152,50.107797,8.289937,12.563293,180.102204,...,NaN,NaN,NaN,NaN,NaN,NaN,9.082285,19.464816,NaN,59.847548
40558_x_intercept_abs,NaN,29.431502,17.432832,NaN,NaN,15.060308,12.708063,0.428948,2.521662,NaN,...,46.803496,3.689947,NaN,NaN,13.823864,NaN,NaN,0.094391,2.248061,52.961229
40559_x_intercept_abs,NaN,29.390637,7.647542,NaN,NaN,NaN,28.061566,12.069783,1.329046,99.308616,...,NaN,13.624511,6.479223,NaN,11.625150,NaN,1.649238,NaN,NaN,68.245241
40560_x_intercept_abs,NaN,NaN,NaN,12.681492,NaN,NaN,NaN,NaN,10.655277,NaN,...,NaN,NaN,11.452695,NaN,9.913994,NaN,15.104056,NaN,NaN,NaN


In [24]:
abs_df.index = range(len(abs_df))

In [19]:
abs_df

,58.0296,71.0503,74.0246,84.0454,99.0077,101.0239,102.0275,114.9259,115.0035,118.0505,...,757.4864,758.5665,775.4865,781.4886,783.5035,784.5070,818.4730,831.4993,856.5052,860.5360
0,NaN,NaN,NaN,NaN,NaN,NaN,54.500993,33.153667,29.568411,NaN,...,NaN,NaN,3.443140,NaN,40.829728,NaN,11.649830,NaN,NaN,113.407464
1,71.521029,NaN,NaN,31.004770,NaN,NaN,32.604357,NaN,7.987146,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,14.406159,NaN,NaN,107.342227
2,37.221524,NaN,22.491404,59.076727,NaN,88.804098,16.879574,NaN,NaN,NaN,...,2.300068,NaN,NaN,NaN,50.001698,NaN,10.836148,NaN,NaN,73.631896
3,NaN,37.664302,18.797747,NaN,43.470226,NaN,NaN,19.012033,5.225182,29.492769,...,NaN,NaN,NaN,22.946539,9.948982,NaN,10.817250,NaN,NaN,41.851645
4,62.844429,NaN,23.308197,NaN,NaN,NaN,28.590537,12.322394,9.863505,NaN,...,12.226012,0.463222,NaN,NaN,NaN,NaN,37.271538,21.730873,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40557,42.243887,NaN,23.491930,NaN,NaN,64.634152,50.107797,8.289937,12.563293,180.102204,...,NaN,NaN,NaN,NaN,NaN,NaN,9.082285,19.464816,NaN,59.847548
40558,NaN,29.431502,17.432832,NaN,NaN,15.060308,12.708063,0.428948,2.521662,NaN,...,46.803496,3.689947,NaN,NaN,13.823864,NaN,NaN,0.094391,2.248061,52.961229
40559,NaN,29.390637,7.647542,NaN,NaN,NaN,28.061566,12.069783,1.329046,99.308616,...,NaN,13.624511,6.479223,NaN,11.625150,NaN,1.649238,NaN,NaN,68.245241
40560,NaN,NaN,NaN,12.681492,NaN,NaN,NaN,NaN,10.655277,NaN,...,NaN,NaN,11.452695,NaN,9.913994,NaN,15.104056,NaN,NaN,NaN


In [184]:
chainedIndexLabel=pd.read_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/chainedIndexLabel.csv")

In [13]:
chainedOCoordsLabel=pd.read_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/chainedOCoordsLabel.csv")

In [14]:
chainedOCoordsLabel=chainedOCoordsLabel.iloc[:,1:]

In [5]:
chainedOCoordsLabel

,0-original-x,0-original-y,15-original-x,15-original-y,30-original-x,30-original-y,45-original-x,45-original-y,60-original-x,60-original-y
0,50479.0,830.0,43299.0,8430.0,29419.0,8610.0,15459.0,9650.0,2019.0,9510.0
1,48279.0,1530.0,45499.0,7470.0,31639.0,8050.0,17739.0,8970.0,4299.0,8890.0
2,52079.0,3850.0,41719.0,5110.0,28079.0,5450.0,14299.0,6330.0,599.0,6570.0
3,52099.0,3850.0,41699.0,5110.0,28059.0,5450.0,14279.0,6350.0,599.0,6590.0
4,52119.0,3850.0,41679.0,5090.0,28039.0,5430.0,14219.0,6270.0,559.0,6550.0
...,...,...,...,...,...,...,...,...,...,...
40557,49319.0,3250.0,44519.0,5610.0,30639.0,6070.0,16779.0,6950.0,3359.0,6830.0
40558,49739.0,1170.0,44079.0,7970.0,31659.0,8050.0,17759.0,9010.0,4359.0,8890.0
40559,49759.0,1170.0,44079.0,7990.0,31659.0,8070.0,17779.0,9030.0,4379.0,8910.0
40560,50039.0,350.0,44059.0,8690.0,30739.0,8830.0,17039.0,9990.0,3639.0,9910.0


In [6]:
chainedOCoordsLabel.columns

Index(['0-original-x', '0-original-y', '15-original-x', '15-original-y',
       '30-original-x', '30-original-y', '45-original-x', '45-original-y',
       '60-original-x', '60-original-y'],
      dtype='object')

In [191]:


abs_df = abs_df.copy()
chainedOCoordsLabel = chainedOCoordsLabel.copy()

abs_df.index = range(abs_df.shape[0])
chainedOCoordsLabel.index = range(chainedOCoordsLabel.shape[0])

samples = [0, 15, 30, 45, 60]

out_dir = '/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/heatmaps_all_mz_log_clip_5-95'
os.makedirs(out_dir, exist_ok=True)

# 这两个参数可以调
low_q = 5      # 下限分位数
high_q = 95    # 上限分位数

for mz in abs_df.columns:
    vals = pd.to_numeric(abs_df[mz], errors='coerce')
    vals = vals[np.isfinite(vals)]

    # 只用正值来确定色标范围
    vals_pos = vals[vals > 0]
    if len(vals_pos) == 0:
        continue

    # 在 log 空间里算分位数
    log_vals = np.log1p(vals_pos)
    vmin = np.nanpercentile(log_vals, low_q)
    vmax = np.nanpercentile(log_vals, high_q)

    if pd.isna(vmin) or pd.isna(vmax) or vmin >= vmax:
        continue

    fig, axes = plt.subplots(1, 5, figsize=(25, 5), constrained_layout=True)
    last_im = None

    for ax, s in zip(axes, samples):
        x_col = f'{s}-original-x'
        y_col = f'{s}-original-y'

        tmp = pd.DataFrame({
            'x': pd.to_numeric(chainedOCoordsLabel[x_col], errors='coerce'),
            'y': pd.to_numeric(chainedOCoordsLabel[y_col], errors='coerce'),
            'conc': pd.to_numeric(abs_df[mz], errors='coerce')
        }).dropna()

        # 只画正值；0值不参与显示
        tmp = tmp[tmp['conc'] > 0]

        if tmp.empty:
            ax.set_title(f'Sample {s}')
            ax.set_axis_off()
            continue

        # log1p 压缩动态范围
        tmp['conc_plot'] = np.log1p(tmp['conc'])

        # 在 log 空间再 clip 一次
        tmp['conc_plot'] = tmp['conc_plot'].clip(lower=vmin, upper=vmax)

        x_unique = np.sort(tmp['x'].unique())
        y_unique = np.sort(tmp['y'].unique())

        heat = tmp.pivot_table(index='y', columns='x', values='conc_plot', aggfunc='mean')
        heat = heat.reindex(index=y_unique, columns=x_unique)

        dx = np.min(np.diff(x_unique)) if len(x_unique) > 1 else 1
        dy = np.min(np.diff(y_unique)) if len(y_unique) > 1 else 1

        Z = np.ma.masked_invalid(heat.values)

        last_im = ax.imshow(
            Z,
            origin='lower',
            aspect='equal',
            extent=[
                x_unique.min() - dx / 2, x_unique.max() + dx / 2,
                y_unique.min() - dy / 2, y_unique.max() + dy / 2
            ],
            vmin=vmin,
            vmax=vmax
        )

        ax.set_title(f'Sample {s}')
        ax.set_xlabel('x')
        ax.set_ylabel('y')

    if last_im is not None:
        fig.colorbar(last_im, ax=axes, shrink=0.8, label='log1p(concentration)')

    plt.suptitle(f'Heatmap of metabolite {mz}', y=1.02)
    save_path = os.path.join(out_dir, f'{mz}.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

In [20]:
abs_df = abs_df.copy()
chainedOCoordsLabel = chainedOCoordsLabel.copy()

abs_df.index = range(abs_df.shape[0])
chainedOCoordsLabel.index = range(chainedOCoordsLabel.shape[0])

samples = [0, 15, 30, 45, 60]

out_dir = '/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/0_heatmaps_all_mz_log_clip_5-95'
os.makedirs(out_dir, exist_ok=True)

# 这两个参数可以调
low_q = 5      # 下限分位数
high_q = 95    # 上限分位数

for mz in abs_df.columns:
    vals = pd.to_numeric(abs_df[mz], errors='coerce')
    vals = vals[np.isfinite(vals)]

    # 只用正值来确定色标范围
    vals_pos = vals[vals > 0]
    if len(vals_pos) == 0:
        continue

    # 在 log 空间里算分位数
    log_vals = np.log1p(vals_pos)
    vmin = np.nanpercentile(log_vals, low_q)
    vmax = np.nanpercentile(log_vals, high_q)

    if pd.isna(vmin) or pd.isna(vmax) or vmin >= vmax:
        continue

    fig, ax = plt.subplots(1, 1, figsize=(5, 5), constrained_layout=True)
    last_im = None

    s = 0
    x_col = f'{s}-original-x'
    y_col = f'{s}-original-y'

    tmp = pd.DataFrame({
        'x': pd.to_numeric(chainedOCoordsLabel[x_col], errors='coerce'),
        'y': pd.to_numeric(chainedOCoordsLabel[y_col], errors='coerce'),
        'conc': pd.to_numeric(abs_df[mz], errors='coerce')
    }).dropna()

    # 只画正值；0值不参与显示
    tmp = tmp[tmp['conc'] > 0]

    if tmp.empty:
        ax.set_title(f'Sample {s}')
        ax.set_axis_off()
    else:
        # log1p 压缩动态范围
        tmp['conc_plot'] = np.log1p(tmp['conc'])

        # 在 log 空间再 clip 一次
        tmp['conc_plot'] = tmp['conc_plot'].clip(lower=vmin, upper=vmax)

        x_unique = np.sort(tmp['x'].unique())
        y_unique = np.sort(tmp['y'].unique())

        heat = tmp.pivot_table(index='y', columns='x', values='conc_plot', aggfunc='mean')
        heat = heat.reindex(index=y_unique, columns=x_unique)

        dx = np.min(np.diff(x_unique)) if len(x_unique) > 1 else 1
        dy = np.min(np.diff(y_unique)) if len(y_unique) > 1 else 1

        Z = np.ma.masked_invalid(heat.values)

        last_im = ax.imshow(
            Z,
            origin='lower',
            aspect='equal',
            extent=[
                x_unique.min() - dx / 2, x_unique.max() + dx / 2,
                y_unique.min() - dy / 2, y_unique.max() + dy / 2
            ],
            vmin=vmin,
            vmax=vmax
        )

        ax.set_title(f'Sample {s}')
        ax.set_xlabel('x')
        ax.set_ylabel('y')

    if last_im is not None:
        fig.colorbar(last_im, ax=ax, shrink=0.8, label='log1p(concentration)')

    plt.suptitle(f'Heatmap of metabolite {mz}', y=1.02)
    save_path = os.path.join(out_dir, f'{mz}.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

In [5]:
cluster_abs_df=pd.read_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_calculateByClusterLR/relativeConcentration.csv")


In [6]:
cluster_abs_df = cluster_abs_df.iloc[:,[0,1,4,5]]

In [7]:
cluster_abs_df

,metabolite,cell_label,r2,x_intercept_abs
0,57.0346,0,0.907794,200.142660
1,57.0346,1,0.899240,431.361201
2,57.0346,2,0.789031,233.726090
3,57.0346,3,0.910105,236.864606
4,57.0346,4,0.957983,204.671025
...,...,...,...,...
13915,986.6105,15,0.921246,125.734535
13916,986.6105,16,0.724794,149.661418
13917,986.6105,17,0.893816,99.364934
13918,986.6105,18,0.851863,115.070806


In [8]:
r207Spots8000=pd.read_csv("/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/r207Spots8000.csv",index_col=0)

In [9]:
r207Spots8000

,0
58.0296,8803
71.0503,12463
74.0246,19951
84.0454,11646
99.0077,9792
...,...
784.5070,8362
818.4730,29161
831.4993,17049
856.5052,8676


In [10]:
cluster_abs_df=cluster_abs_df[cluster_abs_df['metabolite'].isin(r207Spots8000.index)]

In [11]:
cluster_abs_df.loc[cluster_abs_df['r2'] < 0.7, 'x_intercept_abs'] = np.nan

In [12]:
cluster_abs_df

,metabolite,cell_label,r2,x_intercept_abs
20,58.0296,0,0.985569,373.429464
21,58.0296,1,0.868873,303.990925
22,58.0296,2,0.581705,NaN
23,58.0296,3,0.980617,492.687722
24,58.0296,4,0.914470,298.200921
...,...,...,...,...
13775,860.5360,15,0.943496,134.569116
13776,860.5360,16,0.877238,127.403754
13777,860.5360,17,0.905598,85.795759
13778,860.5360,18,0.876211,111.363822


In [4]:
adata = ad.read_h5ad("/p2/zulab/jtian/data/SA/05_CAST/output_cast9/demo1_cast9.h5ad")
cell_label_dict = torch.load("/p1/data/jtian/SA/05_CAST/output_castProj1/cell_label_dict_k20.pt", map_location="cpu")
sample_order = ['0','15','30','45','60']
cell_label = np.concatenate([cell_label_dict[k] for k in sample_order])
adata.obs['cell_label'] = cell_label
labelCoords = adata.obs.copy()

In [37]:
labelCoords.columns

Index(['x', 'y', 'sample', 'cell_label'], dtype='object')

In [39]:
# -------- 复制，避免改原数据 --------
abs_df = abs_df.copy()
chainedOCoordsLabel = chainedOCoordsLabel.copy()
cluster_abs_df = cluster_abs_df.copy()
labelCoords = labelCoords.copy()

# -------- 重置索引，保证 point 图按行对应 --------
abs_df.index = range(abs_df.shape[0])
chainedOCoordsLabel.index = range(chainedOCoordsLabel.shape[0])
cluster_abs_df.index = range(cluster_abs_df.shape[0])
labelCoords.index = range(labelCoords.shape[0])

# -------- 统一类型 --------
cluster_abs_df['cell_label'] = pd.to_numeric(cluster_abs_df['cell_label'], errors='coerce')
cluster_abs_df['x_intercept_abs'] = pd.to_numeric(cluster_abs_df['x_intercept_abs'], errors='coerce')

labelCoords['x'] = pd.to_numeric(labelCoords['x'], errors='coerce')
labelCoords['y'] = pd.to_numeric(labelCoords['y'], errors='coerce')
labelCoords['sample'] = pd.to_numeric(labelCoords['sample'], errors='coerce')
labelCoords['cell_label'] = pd.to_numeric(labelCoords['cell_label'], errors='coerce')

# 只取样本0的 cluster 空间图底图
sample0_coords = labelCoords[labelCoords['sample'] == 0].copy()

# -------- 输出路径 --------
out_dir = '/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/heatmaps_point_vs_cluster_sample0'
os.makedirs(out_dir, exist_ok=True)

# -------- 色标裁剪分位数，可调 --------
low_q = 5
high_q = 95

for mz in abs_df.columns:
    # =========================
    # 1) point-based 数据（样本0）
    # =========================
    point_tmp = pd.DataFrame({
        'x': pd.to_numeric(chainedOCoordsLabel['0-original-x'], errors='coerce'),
        'y': pd.to_numeric(chainedOCoordsLabel['0-original-y'], errors='coerce'),
        'conc': pd.to_numeric(abs_df[mz], errors='coerce')
    }).dropna()

    point_tmp = point_tmp[point_tmp['conc'] > 0].copy()

    # =========================
    # 2) cluster-based 数据（样本0）
    # =========================
    sub = cluster_abs_df.loc[
        cluster_abs_df['metabolite'] == mz,
        ['cell_label', 'x_intercept_abs']
    ].copy()

    sub['cell_label'] = pd.to_numeric(sub['cell_label'], errors='coerce')
    sub['x_intercept_abs'] = pd.to_numeric(sub['x_intercept_abs'], errors='coerce')

    conc_map = sub.set_index('cell_label')['x_intercept_abs']

    cluster_tmp = sample0_coords[['x', 'y', 'cell_label']].copy()
    cluster_tmp['conc'] = cluster_tmp['cell_label'].map(conc_map)
    cluster_tmp = cluster_tmp.dropna(subset=['x', 'y', 'conc'])
    cluster_tmp = cluster_tmp[cluster_tmp['conc'] > 0].copy()

    # =========================
    # 3) 两边共用同一个色标范围
    #    用 point + cluster 的正值合并后计算
    # =========================
    vals_point = point_tmp['conc'].values if not point_tmp.empty else np.array([])
    vals_cluster = cluster_tmp['conc'].values if not cluster_tmp.empty else np.array([])

    vals_all = np.concatenate([vals_point, vals_cluster]) if (len(vals_point) + len(vals_cluster)) > 0 else np.array([])

    vals_all = vals_all[np.isfinite(vals_all)]
    vals_all = vals_all[vals_all > 0]

    if len(vals_all) == 0:
        continue

    log_vals_all = np.log1p(vals_all)
    vmin = np.nanpercentile(log_vals_all, low_q)
    vmax = np.nanpercentile(log_vals_all, high_q)

    if pd.isna(vmin) or pd.isna(vmax) or vmin >= vmax:
        continue

    # =========================
    # 4) 开始画图：一张图里两个子图
    # =========================
    fig, axes = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)
    last_im = None

    # ---------- 左图：point-based ----------
    ax = axes[0]

    if point_tmp.empty:
        ax.set_title('Point-based (Sample 0)')
        ax.set_axis_off()
    else:
        point_tmp['conc_plot'] = np.log1p(point_tmp['conc'])
        point_tmp['conc_plot'] = point_tmp['conc_plot'].clip(lower=vmin, upper=vmax)

        x_unique = np.sort(point_tmp['x'].unique())
        y_unique = np.sort(point_tmp['y'].unique())

        heat = point_tmp.pivot_table(
            index='y',
            columns='x',
            values='conc_plot',
            aggfunc='mean'
        )
        heat = heat.reindex(index=y_unique, columns=x_unique)

        dx = np.min(np.diff(x_unique)) if len(x_unique) > 1 else 1
        dy = np.min(np.diff(y_unique)) if len(y_unique) > 1 else 1

        Z = np.ma.masked_invalid(heat.values)

        last_im = ax.imshow(
            Z,
            origin='lower',
            aspect='equal',
            extent=[
                x_unique.min() - dx / 2, x_unique.max() + dx / 2,
                y_unique.min() - dy / 2, y_unique.max() + dy / 2
            ],
            vmin=vmin,
            vmax=vmax
        )

        ax.set_title('Point-based (Sample 0)')
        ax.set_xlabel('x')
        ax.set_ylabel('y')

    # ---------- 右图：cluster-based ----------
    ax = axes[1]

    if cluster_tmp.empty:
        ax.set_title('Cluster-based (Sample 0)')
        ax.set_axis_off()
    else:
        cluster_tmp['conc_plot'] = np.log1p(cluster_tmp['conc'])
        cluster_tmp['conc_plot'] = cluster_tmp['conc_plot'].clip(lower=vmin, upper=vmax)

        x_unique = np.sort(cluster_tmp['x'].unique())
        y_unique = np.sort(cluster_tmp['y'].unique())

        heat = cluster_tmp.pivot_table(
            index='y',
            columns='x',
            values='conc_plot',
            aggfunc='mean'
        )
        heat = heat.reindex(index=y_unique, columns=x_unique)

        dx = np.min(np.diff(x_unique)) if len(x_unique) > 1 else 1
        dy = np.min(np.diff(y_unique)) if len(y_unique) > 1 else 1

        Z = np.ma.masked_invalid(heat.values)

        last_im = ax.imshow(
            Z,
            origin='lower',
            aspect='equal',
            extent=[
                x_unique.min() - dx / 2, x_unique.max() + dx / 2,
                y_unique.min() - dy / 2, y_unique.max() + dy / 2
            ],
            vmin=vmin,
            vmax=vmax
        )

        ax.set_title('Cluster-based (Sample 0)')
        ax.set_xlabel('x')
        ax.set_ylabel('y')

    # ---------- 共用一个 colorbar ----------
    if last_im is not None:
        fig.colorbar(last_im, ax=axes, shrink=0.8, label='log1p(concentration)')

    plt.suptitle(f'Point vs Cluster heatmap of metabolite {mz}', y=1.02)

    mz_str = str(mz).replace('/', '_')
    save_path = os.path.join(out_dir, f'{mz_str}.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

In [40]:
# -------- 复制，避免改原数据 --------
abs_df = abs_df.copy()
chainedOCoordsLabel = chainedOCoordsLabel.copy()
cluster_abs_df = cluster_abs_df.copy()
labelCoords = labelCoords.copy()

# -------- 重置索引，保证 point 图按行对应 --------
abs_df.index = range(abs_df.shape[0])
chainedOCoordsLabel.index = range(chainedOCoordsLabel.shape[0])
cluster_abs_df.index = range(cluster_abs_df.shape[0])
labelCoords.index = range(labelCoords.shape[0])

# -------- 统一类型 --------
cluster_abs_df['cell_label'] = pd.to_numeric(cluster_abs_df['cell_label'], errors='coerce')
cluster_abs_df['x_intercept_abs'] = pd.to_numeric(cluster_abs_df['x_intercept_abs'], errors='coerce')

labelCoords['x'] = pd.to_numeric(labelCoords['x'], errors='coerce')
labelCoords['y'] = pd.to_numeric(labelCoords['y'], errors='coerce')
labelCoords['sample'] = pd.to_numeric(labelCoords['sample'], errors='coerce')
labelCoords['cell_label'] = pd.to_numeric(labelCoords['cell_label'], errors='coerce')

# 如有需要，也可统一 metabolite 类型，避免 cluster_abs_df 和 abs_df.columns 对不上
# cluster_abs_df['metabolite'] = cluster_abs_df['metabolite'].astype(str)
# abs_df.columns = abs_df.columns.astype(str)

# 只取样本0的 cluster 空间图底图
sample0_coords = labelCoords[labelCoords['sample'] == 0].copy()

# -------- 输出路径 --------
out_dir = '/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/heatmaps_point_vs_cluster_sample0_onlyValid'
os.makedirs(out_dir, exist_ok=True)

for mz in abs_df.columns:
    # =========================
    # 1) point-based 数据（样本0）
    # =========================
    point_mask_tmp = pd.DataFrame({
        'x': pd.to_numeric(chainedOCoordsLabel['0-original-x'], errors='coerce'),
        'y': pd.to_numeric(chainedOCoordsLabel['0-original-y'], errors='coerce'),
        'conc': pd.to_numeric(abs_df[mz], errors='coerce')
    }).dropna(subset=['x', 'y', 'conc'])

    # point图真正绘制的数据
    point_tmp = point_mask_tmp[point_mask_tmp['conc'] > 0].copy()

    # point图非NA坐标掩膜
    valid_xy = point_mask_tmp[['x', 'y']].drop_duplicates()

    # =========================
    # 2) cluster-based 数据（样本0）
    # =========================
    sub = cluster_abs_df.loc[
        cluster_abs_df['metabolite'] == mz,
        ['cell_label', 'x_intercept_abs']
    ].copy()

    sub['cell_label'] = pd.to_numeric(sub['cell_label'], errors='coerce')
    sub['x_intercept_abs'] = pd.to_numeric(sub['x_intercept_abs'], errors='coerce')

    # 若同一 metabolite + cell_label 有重复，取平均
    sub = sub.groupby('cell_label', as_index=False)['x_intercept_abs'].mean()

    conc_map = sub.set_index('cell_label')['x_intercept_abs']

    cluster_tmp = sample0_coords[['x', 'y', 'cell_label']].copy()
    cluster_tmp['conc'] = cluster_tmp['cell_label'].map(conc_map)
    cluster_tmp = cluster_tmp.dropna(subset=['x', 'y', 'conc'])

    # 关键：只保留 point 图非NA坐标
    cluster_tmp = cluster_tmp.merge(valid_xy, on=['x', 'y'], how='inner')

    # 如需和 point 图一致，只画正值
    cluster_tmp = cluster_tmp[cluster_tmp['conc'] > 0].copy()

    # =========================
    # 3) 共用色标：用掩膜后的两边共同计算
    # =========================
    vals_point = point_tmp['conc'].values if not point_tmp.empty else np.array([])
    vals_cluster = cluster_tmp['conc'].values if not cluster_tmp.empty else np.array([])

    vals_all = np.concatenate([vals_point, vals_cluster]) if (len(vals_point) + len(vals_cluster)) > 0 else np.array([])
    vals_all = vals_all[np.isfinite(vals_all)]
    vals_all = vals_all[vals_all > 0]

    if len(vals_all) == 0:
        continue

    log_vals_all = np.log1p(vals_all)
    vmin = np.nanpercentile(log_vals_all, 10)
    vmax = np.nanpercentile(log_vals_all, 90)

    if pd.isna(vmin) or pd.isna(vmax) or vmin >= vmax:
        continue

    # =========================
    # 4) 开始画图：一张图里两个子图
    # =========================
    fig, axes = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)
    last_im = None

    # ---------- 左图：point-based ----------
    ax = axes[0]

    if point_tmp.empty:
        ax.set_title('Point-based (Sample 0)')
        ax.set_axis_off()
    else:
        point_tmp['conc_plot'] = np.log1p(point_tmp['conc'])
        point_tmp['conc_plot'] = point_tmp['conc_plot'].clip(lower=vmin, upper=vmax)

        x_unique = np.sort(point_tmp['x'].unique())
        y_unique = np.sort(point_tmp['y'].unique())

        heat = point_tmp.pivot_table(
            index='y',
            columns='x',
            values='conc_plot',
            aggfunc='mean'
        )
        heat = heat.reindex(index=y_unique, columns=x_unique)

        dx = np.min(np.diff(x_unique)) if len(x_unique) > 1 else 1
        dy = np.min(np.diff(y_unique)) if len(y_unique) > 1 else 1

        Z = np.ma.masked_invalid(heat.values)

        last_im = ax.imshow(
            Z,
            origin='lower',
            aspect='equal',
            extent=[
                x_unique.min() - dx / 2, x_unique.max() + dx / 2,
                y_unique.min() - dy / 2, y_unique.max() + dy / 2
            ],
            vmin=vmin,
            vmax=vmax
        )

        ax.set_title('Point-based (Sample 0)')
        ax.set_xlabel('x')
        ax.set_ylabel('y')

    # ---------- 右图：cluster-based ----------
    ax = axes[1]

    if cluster_tmp.empty:
        ax.set_title('Cluster-based (Sample 0)')
        ax.set_axis_off()
    else:
        cluster_tmp['conc_plot'] = np.log1p(cluster_tmp['conc'])
        cluster_tmp['conc_plot'] = cluster_tmp['conc_plot'].clip(lower=vmin, upper=vmax)

        x_unique = np.sort(cluster_tmp['x'].unique())
        y_unique = np.sort(cluster_tmp['y'].unique())

        heat = cluster_tmp.pivot_table(
            index='y',
            columns='x',
            values='conc_plot',
            aggfunc='mean'
        )
        heat = heat.reindex(index=y_unique, columns=x_unique)

        dx = np.min(np.diff(x_unique)) if len(x_unique) > 1 else 1
        dy = np.min(np.diff(y_unique)) if len(y_unique) > 1 else 1

        Z = np.ma.masked_invalid(heat.values)

        last_im = ax.imshow(
            Z,
            origin='lower',
            aspect='equal',
            extent=[
                x_unique.min() - dx / 2, x_unique.max() + dx / 2,
                y_unique.min() - dy / 2, y_unique.max() + dy / 2
            ],
            vmin=vmin,
            vmax=vmax
        )

        ax.set_title('Cluster-based (Sample 0)')
        ax.set_xlabel('x')
        ax.set_ylabel('y')

    # ---------- 共用一个 colorbar ----------
    if last_im is not None:
        fig.colorbar(last_im, ax=axes, shrink=0.8, label='log1p(concentration)')

    plt.suptitle(f'Point vs Cluster heatmap of metabolite {mz}', y=1.02)

    mz_str = str(mz).replace('/', '_')
    save_path = os.path.join(out_dir, f'{mz_str}.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

In [41]:
# -------- 复制，避免改原数据 --------
abs_df = abs_df.copy()
chainedOCoordsLabel = chainedOCoordsLabel.copy()
cluster_abs_df = cluster_abs_df.copy()
labelCoords = labelCoords.copy()

# -------- 重置索引，保证 point 图按行对应 --------
abs_df.index = range(abs_df.shape[0])
chainedOCoordsLabel.index = range(chainedOCoordsLabel.shape[0])
cluster_abs_df.index = range(cluster_abs_df.shape[0])
labelCoords.index = range(labelCoords.shape[0])

# -------- 统一类型 --------
cluster_abs_df['cell_label'] = pd.to_numeric(cluster_abs_df['cell_label'], errors='coerce')
cluster_abs_df['x_intercept_abs'] = pd.to_numeric(cluster_abs_df['x_intercept_abs'], errors='coerce')

labelCoords['x'] = pd.to_numeric(labelCoords['x'], errors='coerce')
labelCoords['y'] = pd.to_numeric(labelCoords['y'], errors='coerce')
labelCoords['sample'] = pd.to_numeric(labelCoords['sample'], errors='coerce')
labelCoords['cell_label'] = pd.to_numeric(labelCoords['cell_label'], errors='coerce')

# 如有需要，也可统一 metabolite 类型，避免 cluster_abs_df 和 abs_df.columns 对不上
# cluster_abs_df['metabolite'] = cluster_abs_df['metabolite'].astype(str)
# abs_df.columns = abs_df.columns.astype(str)

# 只取样本0的 cluster 空间图底图
sample0_coords = labelCoords[labelCoords['sample'] == 0].copy()

# -------- 输出路径 --------
out_dir = '/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/heatmaps_point_vs_cluster_sample0_separate_scale'
os.makedirs(out_dir, exist_ok=True)

# -------- 各自色标裁剪分位数，可调 --------
low_q = 10
high_q = 90

for mz in abs_df.columns:
    # =========================
    # 1) point-based 数据（样本0）
    # =========================
    point_mask_tmp = pd.DataFrame({
        'x': pd.to_numeric(chainedOCoordsLabel['0-original-x'], errors='coerce'),
        'y': pd.to_numeric(chainedOCoordsLabel['0-original-y'], errors='coerce'),
        'conc': pd.to_numeric(abs_df[mz], errors='coerce')
    }).dropna(subset=['x', 'y', 'conc'])

    # point图真正绘制的数据
    point_tmp = point_mask_tmp[point_mask_tmp['conc'] > 0].copy()

    # point图非NA坐标掩膜
    valid_xy = point_mask_tmp[['x', 'y']].drop_duplicates()

    # =========================
    # 2) cluster-based 数据（样本0）
    # =========================
    sub = cluster_abs_df.loc[
        cluster_abs_df['metabolite'] == mz,
        ['cell_label', 'x_intercept_abs']
    ].copy()

    sub['cell_label'] = pd.to_numeric(sub['cell_label'], errors='coerce')
    sub['x_intercept_abs'] = pd.to_numeric(sub['x_intercept_abs'], errors='coerce')

    # 若同一 metabolite + cell_label 有重复，取平均
    sub = sub.groupby('cell_label', as_index=False)['x_intercept_abs'].mean()

    conc_map = sub.set_index('cell_label')['x_intercept_abs']

    cluster_tmp = sample0_coords[['x', 'y', 'cell_label']].copy()
    cluster_tmp['conc'] = cluster_tmp['cell_label'].map(conc_map)
    cluster_tmp = cluster_tmp.dropna(subset=['x', 'y', 'conc'])

    # 关键：只保留 point 图非NA坐标
    cluster_tmp = cluster_tmp.merge(valid_xy, on=['x', 'y'], how='inner')

    # 如需和 point 图一致，只画正值
    cluster_tmp = cluster_tmp[cluster_tmp['conc'] > 0].copy()

    # 如果两边都没有可画数据，则跳过
    if point_tmp.empty and cluster_tmp.empty:
        continue

    # =========================
    # 3) 分别计算各自色标范围
    # =========================
    point_vmin, point_vmax = None, None
    cluster_vmin, cluster_vmax = None, None

    if not point_tmp.empty:
        point_vals = point_tmp['conc'].values
        point_vals = point_vals[np.isfinite(point_vals)]
        point_vals = point_vals[point_vals > 0]

        if len(point_vals) > 0:
            point_log_vals = np.log1p(point_vals)
            point_vmin = np.nanpercentile(point_log_vals, low_q)
            point_vmax = np.nanpercentile(point_log_vals, high_q)

            if pd.isna(point_vmin) or pd.isna(point_vmax) or point_vmin >= point_vmax:
                point_vmin, point_vmax = None, None

    if not cluster_tmp.empty:
        cluster_vals = cluster_tmp['conc'].values
        cluster_vals = cluster_vals[np.isfinite(cluster_vals)]
        cluster_vals = cluster_vals[cluster_vals > 0]

        if len(cluster_vals) > 0:
            cluster_log_vals = np.log1p(cluster_vals)
            cluster_vmin = np.nanpercentile(cluster_log_vals, low_q)
            cluster_vmax = np.nanpercentile(cluster_log_vals, high_q)

            if pd.isna(cluster_vmin) or pd.isna(cluster_vmax) or cluster_vmin >= cluster_vmax:
                cluster_vmin, cluster_vmax = None, None

    # 如果两边色标都无效，则跳过
    if point_vmin is None and cluster_vmin is None:
        continue

    # =========================
    # 4) 开始画图：一张图里两个子图
    # =========================
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

    point_im = None
    cluster_im = None

    # ---------- 左图：point-based ----------
    ax = axes[0]

    if point_tmp.empty or point_vmin is None:
        ax.set_title('Point-based (Sample 0)')
        ax.set_axis_off()
    else:
        point_tmp['conc_plot'] = np.log1p(point_tmp['conc'])
        point_tmp['conc_plot'] = point_tmp['conc_plot'].clip(lower=point_vmin, upper=point_vmax)

        x_unique = np.sort(point_tmp['x'].unique())
        y_unique = np.sort(point_tmp['y'].unique())

        heat = point_tmp.pivot_table(
            index='y',
            columns='x',
            values='conc_plot',
            aggfunc='mean'
        )
        heat = heat.reindex(index=y_unique, columns=x_unique)

        dx = np.min(np.diff(x_unique)) if len(x_unique) > 1 else 1
        dy = np.min(np.diff(y_unique)) if len(y_unique) > 1 else 1

        Z = np.ma.masked_invalid(heat.values)

        point_im = ax.imshow(
            Z,
            origin='lower',
            aspect='equal',
            extent=[
                x_unique.min() - dx / 2, x_unique.max() + dx / 2,
                y_unique.min() - dy / 2, y_unique.max() + dy / 2
            ],
            vmin=point_vmin,
            vmax=point_vmax
        )

        ax.set_title('Point-based (Sample 0)')
        ax.set_xlabel('x')
        ax.set_ylabel('y')

        fig.colorbar(point_im, ax=ax, shrink=0.8, label='log1p(concentration)')

    # ---------- 右图：cluster-based ----------
    ax = axes[1]

    if cluster_tmp.empty or cluster_vmin is None:
        ax.set_title('Cluster-based (Sample 0)')
        ax.set_axis_off()
    else:
        cluster_tmp['conc_plot'] = np.log1p(cluster_tmp['conc'])
        cluster_tmp['conc_plot'] = cluster_tmp['conc_plot'].clip(lower=cluster_vmin, upper=cluster_vmax)

        x_unique = np.sort(cluster_tmp['x'].unique())
        y_unique = np.sort(cluster_tmp['y'].unique())

        heat = cluster_tmp.pivot_table(
            index='y',
            columns='x',
            values='conc_plot',
            aggfunc='mean'
        )
        heat = heat.reindex(index=y_unique, columns=x_unique)

        dx = np.min(np.diff(x_unique)) if len(x_unique) > 1 else 1
        dy = np.min(np.diff(y_unique)) if len(y_unique) > 1 else 1

        Z = np.ma.masked_invalid(heat.values)

        cluster_im = ax.imshow(
            Z,
            origin='lower',
            aspect='equal',
            extent=[
                x_unique.min() - dx / 2, x_unique.max() + dx / 2,
                y_unique.min() - dy / 2, y_unique.max() + dy / 2
            ],
            vmin=cluster_vmin,
            vmax=cluster_vmax
        )

        ax.set_title('Cluster-based (Sample 0)')
        ax.set_xlabel('x')
        ax.set_ylabel('y')

        fig.colorbar(cluster_im, ax=ax, shrink=0.8, label='log1p(concentration)')

    plt.suptitle(f'Point vs Cluster heatmap of metabolite {mz}', y=1.02)

    mz_str = str(mz).replace('/', '_')
    save_path = os.path.join(out_dir, f'{mz_str}.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

In [25]:
# -------- 复制，避免改原数据 --------
abs_df = abs_df.copy()
chainedOCoordsLabel = chainedOCoordsLabel.copy()
cluster_abs_df = cluster_abs_df.copy()
labelCoords = labelCoords.copy()

# -------- 重置索引 --------
abs_df.index = range(abs_df.shape[0])
chainedOCoordsLabel.index = range(chainedOCoordsLabel.shape[0])
cluster_abs_df.index = range(cluster_abs_df.shape[0])
labelCoords.index = range(labelCoords.shape[0])

# -------- 统一类型 --------
cluster_abs_df['cell_label'] = pd.to_numeric(cluster_abs_df['cell_label'], errors='coerce')
cluster_abs_df['x_intercept_abs'] = pd.to_numeric(cluster_abs_df['x_intercept_abs'], errors='coerce')

labelCoords['x'] = pd.to_numeric(labelCoords['x'], errors='coerce')
labelCoords['y'] = pd.to_numeric(labelCoords['y'], errors='coerce')
labelCoords['sample'] = pd.to_numeric(labelCoords['sample'], errors='coerce')
labelCoords['cell_label'] = pd.to_numeric(labelCoords['cell_label'], errors='coerce')

# 如有需要，也可统一 metabolite 类型，避免 cluster_abs_df 和 abs_df.columns 对不上
# cluster_abs_df['metabolite'] = cluster_abs_df['metabolite'].astype(str)
# abs_df.columns = abs_df.columns.astype(str)

# 只取样本0
sample0_coords = labelCoords[labelCoords['sample'] == 0].copy()

# -------- 输出路径 --------
out_dir = '/p2/zulab/jtian/data/SA/06_calculateConcentration/output_findChain/heatmaps_Onlycluster_sample0_10-90'
os.makedirs(out_dir, exist_ok=True)

# -------- 色标裁剪分位数，可调 --------
low_q = 10
high_q = 90

for mz in abs_df.columns:
    # =========================
    # 1) cluster-based 数据（样本0，保留所有坐标）
    # =========================
    sub = cluster_abs_df.loc[
        cluster_abs_df['metabolite'] == mz,
        ['cell_label', 'x_intercept_abs']
    ].copy()

    sub['cell_label'] = pd.to_numeric(sub['cell_label'], errors='coerce')
    sub['x_intercept_abs'] = pd.to_numeric(sub['x_intercept_abs'], errors='coerce')

    # 若同一 metabolite + cell_label 有重复，取平均
    sub = sub.groupby('cell_label', as_index=False)['x_intercept_abs'].mean()

    conc_map = sub.set_index('cell_label')['x_intercept_abs']

    cluster_tmp = sample0_coords[['x', 'y', 'cell_label']].copy()
    cluster_tmp['conc'] = cluster_tmp['cell_label'].map(conc_map)
    cluster_tmp = cluster_tmp.dropna(subset=['x', 'y', 'conc'])

    # 只画正值
    cluster_tmp = cluster_tmp[cluster_tmp['conc'] > 0].copy()

    if cluster_tmp.empty:
        continue

    # =========================
    # 2) 计算 cluster 自己的色标范围
    # =========================
    cluster_vals = cluster_tmp['conc'].values
    cluster_vals = cluster_vals[np.isfinite(cluster_vals)]
    cluster_vals = cluster_vals[cluster_vals > 0]

    if len(cluster_vals) == 0:
        continue

    cluster_log_vals = np.log1p(cluster_vals)
    cluster_vmin = np.nanpercentile(cluster_log_vals, low_q)
    cluster_vmax = np.nanpercentile(cluster_log_vals, high_q)

    if pd.isna(cluster_vmin) or pd.isna(cluster_vmax) or cluster_vmin >= cluster_vmax:
        continue

    # =========================
    # 3) 只画 cluster 图
    # =========================
    fig, ax = plt.subplots(1, 1, figsize=(6, 5), constrained_layout=True)

    cluster_tmp['conc_plot'] = np.log1p(cluster_tmp['conc'])
    cluster_tmp['conc_plot'] = cluster_tmp['conc_plot'].clip(lower=cluster_vmin, upper=cluster_vmax)

    x_unique = np.sort(cluster_tmp['x'].unique())
    y_unique = np.sort(cluster_tmp['y'].unique())

    heat = cluster_tmp.pivot_table(
        index='y',
        columns='x',
        values='conc_plot',
        aggfunc='mean'
    )
    heat = heat.reindex(index=y_unique, columns=x_unique)

    dx = np.min(np.diff(x_unique)) if len(x_unique) > 1 else 1
    dy = np.min(np.diff(y_unique)) if len(y_unique) > 1 else 1

    Z = np.ma.masked_invalid(heat.values)

    cluster_im = ax.imshow(
        Z,
        origin='lower',
        aspect='equal',
        extent=[
            x_unique.min() - dx / 2, x_unique.max() + dx / 2,
            y_unique.min() - dy / 2, y_unique.max() + dy / 2
        ],
        vmin=cluster_vmin,
        vmax=cluster_vmax
    )

    ax.set_title('Cluster-based (Sample 0)')
    ax.set_xlabel('x')
    ax.set_ylabel('y')

    fig.colorbar(cluster_im, ax=ax, shrink=0.8, label='log1p(concentration)')

    plt.suptitle(f'Cluster heatmap of metabolite {mz}', y=1.02)

    mz_str = str(mz).replace('/', '_')
    save_path = os.path.join(out_dir, f'{mz_str}.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)